# XLAV — E4: LoRA cho CLIP vision encoder của CLIPSeg

Các thử nghiệm thay decoder (thêm cross-attention theo từ, tăng `reduce_dim`) không vượt đối chứng,
nên E4 chuyển sang **module encoder**: gắn **LoRA** vào `q_proj`/`v_proj` của các layer CLIP vision
có output đi vào decoder (layer 0–9). Trọng số CLIP gốc vẫn đóng băng; chỉ LoRA (~0,25M tham số) và decoder được học.
Loss (BCE), dữ liệu PhraseCut+ và protocol chấm giữ nguyên như E1 để so được với tác giả.

| `ARM` | Model | Vai trò |
|---|---|---|
| `"B1"` | E1 giữ nguyên kiến trúc | Đối chứng: tách tác động của việc "train thêm" |
| `"E4c"` | E1 + LoRA (rank 8) trên CLIP vision | Cải tiến |

**Mặc định (`FROM_SCRATCH = True`): E4c train lại từ đầu** — decoder ngẫu nhiên (cùng seed với E1) và LoRA có mặt
ngay từ epoch 1 — trong **9 epoch**, đúng lịch mà B1 đã nhận nếu tính từ đầu:

| Chặng | Epoch | Learning rate (cosine) | Như |
|---|---|---|---|
| 1 | 1–5 | 1e-3 → 1e-4 | E1 lượt đầu |
| 2 | 6–7 | 1e-4 → 5e-5, giữ optimizer | E1 train tiếp |
| 3 | 8–9 | optimizer mới, 3e-4 → 1e-5 | B1 |

Vì một phiên Kaggle tối đa 12 giờ, chia thành nhiều phiên bằng `STOP_AFTER_EPOCH` (ví dụ 3 → 6 → hết).
Phiên sau tự tìm checkpoint mới nhất của phiên trước trong Input (`RESUME_CHECKPOINT = "auto"`), kiểm tra
cùng cấu hình rồi chạy tiếp, giữ optimizer, scheduler, AMP scaler và RNG. Chỉ phiên cuối (epoch 9) chấm val/test.

`FROM_SCRATCH = False` (cách nhanh): khởi tạo từ E1 epoch 7 và train thêm `EPOCHS` epoch như B1;
LoRA khởi tạo `B = 0` nên lúc bắt đầu cho **đúng** logits của E1 (notebook kiểm tra).
Embedding điều kiện (text và ảnh support) luôn tính bằng CLIP gốc (tắt LoRA), nên LoRA chỉ tinh chỉnh
đặc trưng dày của ảnh query — visual prompt ở E2 không bị đổi nghĩa.

Cả hai cách đều so E4c với **B1** (cùng tổng 9 epoch). Chọn trên **val**; test chỉ để báo cáo.

**Chạy trên Kaggle**
1. Add Input: dataset PhraseCut (`poudelsandeep/phrasecut`). Phiên 2 trở đi: thêm Output của phiên trước.
   (Chỉ `FROM_SCRATCH = False` mới cần dataset checkpoint E1 epoch 7, SHA256 `b9bc2850…`.)
2. Settings: **Accelerator GPU T4 x2**, **Internet on**. Đặt `STOP_AFTER_EPOCH`, rồi **Save & Run All**.
3. Kết quả ở `/kaggle/working/clipseg_e4_<arm>` và file `clipseg_e4_<arm>.zip` (có checkpoint từng epoch).


## 1. Dependency

Giữ PyTorch/CUDA, NumPy, Pillow, scikit-learn và matplotlib sẵn có của Kaggle.
Cell này chỉ pin Transformers và gdown; không cài wheel PyTorch CPU.
Nếu kernel đã import Transformers phiên bản khác trước đó, restart session rồi chạy lại từ đầu.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.55.4", "gdown==5.2.0",
])


## 2. Cấu hình

Mặc định tải dữ liệu online. Nếu đã Add Input một Kaggle Dataset chứa dữ liệu,
đặt `DATA_ROOT` tới thư mục chứa `refer_<EVAL_SPLIT>.json`, `image_data_split.json`
(hoặc `image_data_split3000.json`) và `images/`. Để `None` sẽ tự tìm trong `/kaggle/input`
rồi tải từ nguồn chính thức nếu chưa có. `MODEL_PATH=None` tải Hugging Face;
muốn offline thì trỏ tới thư mục snapshot có config, processor, tokenizer và weights.

Baseline mặc định FP32, dùng một GPU; T4 x2 vẫn chạy trên GPU đầu tiên.
`USE_AMP=True` là tùy chọn FP16, phải ghi riêng khi so sánh số liệu.


In [ ]:
from pathlib import Path

ARM = "E4c"                      # "E4c" = LoRA trên CLIP vision (cải tiến) | "B1" = đối chứng train tiếp kiến trúc gốc
INIT_CHECKPOINT = None           # checkpoint E1 (decoder_epoch_007.pt/resume_latest.pt); None = tự tìm trong Kaggle Input
INIT_CHECKPOINT_SHA256 = "b9bc28509a6a6f36495bc4826a511d182bdee8885a819190997da56fd67947df"  # E1 epoch 7; None = không kiểm
LORA_RANK = 8                    # E4c: hạng LoRA trên q_proj/v_proj của các layer CLIP vision mà decoder dùng
LORA_ALPHA = 16                  # E4c: hệ số LoRA, scale = alpha / rank
LORA_GRAD_CHECKPOINT = False     # E4c: True nếu báo thiếu VRAM (CUDA out of memory); chậm hơn ~30%
FROM_SCRATCH = True              # True = train lại từ đầu 9 epoch theo lịch của B1 | False = train tiếp từ E1 epoch 7
STOP_AFTER_EPOCH = None          # FROM_SCRATCH: dừng phiên sau epoch này để nối phiên (vd 3, rồi 6); None = chạy tới epoch 9
LORA_LR_SCALE = 1.0              # LR của LoRA = LORA_LR_SCALE × LR decoder, cùng lịch cosine (vd 0.1, 0.03)
TEST_TIME_BUDGET_HOURS = None    # bỏ chấm test nếu phiên đã chạy quá số giờ này khi chấm xong val (vd 10.0); None = luôn chấm

DATA_ROOT = None                 # ví dụ: "/kaggle/input/phrasecut-miniv/VGPhraseCut_v0"
MODEL_PATH = None                # thư mục checkpoint offline, hoặc None
IMAGE_ROOT = None                # thư mục chứa ảnh <image_id>.jpg; None = tìm trong mọi Kaggle Input
WORK_ROOT = Path("/kaggle/working")
REQUIRE_GPU = True               # False chỉ khi kiểm tra trên máy CPU
EVAL_SPLIT = "val"             # "miniv" (100 ảnh) | "val" (~2.900 ảnh) | "test" (2.545 ảnh, split của paper)
MAX_QUERIES = None               # None = toàn bộ split; 8 = smoke
BATCH_SIZE = 64
EPOCHS = 2
TRAIN_DECODER = True            # E1 = False: chỉ đánh giá checkpoint pretrained
TRAIN_ROOT = None                # thư mục refer_train.json + images/ trong Kaggle Input
MAX_TRAIN_QUERIES = None         # None = toàn bộ train; 8 = smoke train
TRAIN_INIT = "e1_checkpoint"   # "pretrained_decoder" = fine-tune decoder đã học
TRAIN_MODE = "phrasecut_plus"    # "phrasecut_plus" = text + ảnh support như paper; "text_only" = ablation rd64-uni-novis
NEGATIVE_PROB = 0.2              # tỉ lệ mẫu âm: cụm từ không có trong ảnh, mask rỗng
MIX_TEXT_MAX = 0.5               # trọng số text w ~ U(0, 0,5); mẫu không có support dùng w = 1
PROMPT_AUGMENT = True            # train với 9 template "shuffle+" của tác giả
AUG_CROP = True                  # crop vuông ngẫu nhiên, giữ vật trong khung
LEARNING_RATE = 3e-4
MIN_LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
NUM_WORKERS = 4                  # worker tạo mẫu PhraseCut+ (Kaggle có 4 CPU); 0 khi chạy trên Windows
MULTI_GPU = True                 # train chia mỗi batch cho mọi GPU (T4 x2) bằng DataParallel
RESUME_CHECKPOINT = None         # checkpoint epoch của chính notebook, hoặc None
USE_AMP = True                  # FP16 autocast khi TRAIN (nên True trên T4)
EVAL_AMP = False                 # đánh giá FP32 như score.py của tác giả
DECODER_CHECKPOINT = None        # TRAIN_DECODER=False: đánh giá decoder_epoch_XXX.pt đã train thay cho decoder tác giả
SEED = 42
THRESHOLD = 0.3
CALIBRATION_FRACTION = 0.2
THRESHOLD_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
PROMPT_TEMPLATE = "{phrase}"      # giữ nguyên phrase, cố định trước evaluation
SAVE_PROBABILITIES = False       # True lưu float32 .npy (tốn thêm dung lượng)
N_VISUALIZATIONS = 6
DOWNLOAD_WORKERS = 4

MODEL_ID = "CIDAS/clipseg-rd64-refined"
MODEL_REVISION = "999e0328d9e10b484360c477313983f9afdd7050"
PHRASECUT_API_COMMIT = "276470b7773d8a230aee5b5536876fbc485fb04b"

import time as _time
SESSION_STARTED = _time.time()   # để canh giới hạn 12 giờ của một phiên Kaggle
STOPPED_EARLY = False
RUN_TEST = True
if FROM_SCRATCH:
    # Lịch B1 tính từ đầu: E1 (epoch 1-5 cosine 1e-3 -> 1e-4; epoch 6-7 cosine 1e-4 -> 5e-5, giữ optimizer)
    # rồi B1 (epoch 8-9: optimizer, AMP scaler, thứ tự dữ liệu mới, cosine 3e-4 -> 1e-5).
    SCRATCH_STAGES = [{"epochs": 5, "lr": 1e-3, "min_lr": 1e-4, "start": "random decoder + LoRA"},
                      {"epochs": 2, "lr": 1e-4, "min_lr": 5e-5, "start": "keep optimizer, new cosine"},
                      {"epochs": 2, "lr": 3e-4, "min_lr": 1e-5, "start": "new optimizer, scaler, loader seed"}]
    TRAIN_INIT = "random_decoder"
    EPOCHS = sum(stage["epochs"] for stage in SCRATCH_STAGES)
    LEARNING_RATE, MIN_LEARNING_RATE = SCRATCH_STAGES[0]["lr"], SCRATCH_STAGES[0]["min_lr"]
    if RESUME_CHECKPOINT is None:
        RESUME_CHECKPOINT = "auto"   # tìm checkpoint của phiên trước trong Kaggle Input


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import shutil
import time
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext

import gdown
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor
from IPython.display import display

assert BATCH_SIZE >= 1
assert isinstance(EPOCHS, int) and EPOCHS >= 1
assert TRAIN_INIT in ["random_decoder", "pretrained_decoder", "e1_checkpoint"]
assert ARM in ["B1", "E4c"] and LORA_RANK >= 1
assert not FROM_SCRATCH or ARM == "E4c", "B1 từ đầu chính là E1 + B1, đã có"
assert STOP_AFTER_EPOCH is None or (FROM_SCRATCH and 1 <= STOP_AFTER_EPOCH <= EPOCHS)
assert TRAIN_MODE in ["phrasecut_plus", "text_only"]
assert EVAL_SPLIT in ["miniv", "val", "test"]
assert not (TRAIN_DECODER and DECODER_CHECKPOINT), "DECODER_CHECKPOINT chỉ dùng khi TRAIN_DECODER=False."
assert 0 <= NEGATIVE_PROB < 1 and 0 <= MIX_TEXT_MAX <= 1
assert MAX_TRAIN_QUERIES is None or MAX_TRAIN_QUERIES >= 1
assert 0 <= MIN_LEARNING_RATE <= LEARNING_RATE and LEARNING_RATE > 0
assert MAX_QUERIES is None or MAX_QUERIES >= 1
assert 0 < CALIBRATION_FRACTION < 1
assert 0 <= THRESHOLD <= 1 and all(0 <= t <= 1 for t in THRESHOLD_GRID)
assert "{phrase}" in PROMPT_TEMPLATE
random.seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA. Bật GPU trong Kaggle Settings rồi restart session.")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    try:
        for index in range(torch.cuda.device_count()):
            torch.ones(1, device=f"cuda:{index}").add_(1).item()
    except RuntimeError as exc:
        raise RuntimeError("PyTorch của Kaggle không chạy được trên GPU này (thường gặp với P100). "
                           "Chọn Accelerator GPU T4 x2 rồi restart session.") from exc
assert not (USE_AMP or EVAL_AMP) or device.type == "cuda", "FP16 AMP yêu cầu CUDA."
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUT = WORK_ROOT / f"clipseg_e4_{ARM.lower()}"
OUT.mkdir(parents=True, exist_ok=True)
# Xóa dấu hoàn thành cũ trước khi bắt đầu lượt mới.
(OUT / "summary.json").unlink(missing_ok=True)
(OUT / "run.json").unlink(missing_ok=True)
(WORK_ROOT / f"clipseg_e4_{ARM.lower()}.zip").unlink(missing_ok=True)

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def write_csv(path, rows):
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

versions = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "numpy", "Pillow", "scikit-learn", "matplotlib", "gdown"]}
print("Device:", device, "| GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
      if device.type == "cuda" else "CPU")
print(json.dumps(versions, indent=2))


## 3. Lấy annotation chính thức và chọn query

Chỉ tải `refer_<EVAL_SPLIT>.json` và metadata ảnh từ Drive ID trong
[downloader tác giả](https://github.com/ChenyunWu/PhraseCutDataset/blob/276470b7773d8a230aee5b5536876fbc485fb04b/download_dataset.py).
Chỉ ảnh của các query được chọn mới được tải, không tải toàn bộ Visual Genome. Với `test`/`val`
(khoảng 2.500–2.900 ảnh) nên Add Input dataset đã có ảnh để khỏi tải lại mỗi lượt.
Nếu Google Drive hết quota, Add Input dữ liệu đã tải rồi đặt `DATA_ROOT`.

Chia calibration/evaluation theo **image_id**, seed 42, trước khi đọc prediction.
Nếu giới hạn số query, sampling bằng seed, lưu đúng danh sách vào `selected_tasks.json`.


In [ ]:
# Drive ID của annotation chính thức, theo download_dataset.py của PhraseCut.
REFER_DRIVE_IDS = {"miniv": "1oLcTQ1blTIQyu5ZMelQN2HSniuQaxM4E", "val": "1UyojArOFPlsSeNbA9fHWjCjOOU-OCohG",
                   "test": "1jrzXm1gcq6f5hNDeamZd0UmyyHUv61IZ", "train": "1qx-0q6r9r0YUGpoyT0B8HJKmUFWQDSu7"}

def metadata_path(root):
    for name in ["image_data_split.json", "image_data_split3000.json"]:
        candidate = root / name
        if candidate.is_file():
            return candidate
    return root / "image_data_split.json"

if DATA_ROOT is None:
    candidates = sorted(Path("/kaggle/input").rglob(f"refer_{EVAL_SPLIT}.json")) if Path("/kaggle/input").exists() else []
    candidates = [p.parent for p in candidates if metadata_path(p.parent).is_file()]
    if len(candidates) > 1:
        raise ValueError(f"Có nhiều dataset chứa refer_{EVAL_SPLIT}.json; đặt DATA_ROOT rõ ràng: " + str(candidates))
    data_root = candidates[0] if candidates else WORK_ROOT / "phrasecut_annotations"
else:
    data_root = Path(DATA_ROOT)
    if not data_root.is_dir():
        raise FileNotFoundError(f"DATA_ROOT không tồn tại: {data_root}")
data_root.mkdir(parents=True, exist_ok=True)
annotation_path = data_root / f"refer_{EVAL_SPLIT}.json"
info_path = metadata_path(data_root)

def ensure_json(path, file_id):
    if path.is_file():
        value = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError(f"JSON không phải danh sách annotation: {path}")
        return value
    if str(path).startswith("/kaggle/input/"):
        raise FileNotFoundError(f"Thiếu {path.name} trong input read-only; bổ sung Kaggle Dataset.")
    temp = path.with_suffix(".download")
    try:
        result = gdown.download(id=file_id, output=str(temp), quiet=False)
        if result is None:
            raise RuntimeError("Google Drive không trả về file.")
        value = json.loads(temp.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError("Nội dung download không phải annotation JSON.")
        temp.replace(path)
        return value
    except Exception as exc:
        raise RuntimeError(f"Không tải được {path.name}. Bật Internet, hoặc Add Input dataset và đặt DATA_ROOT.") from exc

tasks = ensure_json(annotation_path, REFER_DRIVE_IDS[EVAL_SPLIT])
image_info = ensure_json(info_path, "1xB9eExJo35K3DQu8PnBWZ1q_OMoMl8-i")
info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == EVAL_SPLIT}
assert len({str(t["task_id"]) for t in tasks}) == len(tasks), "Task ID bị trùng."
assert all(int(t["image_id"]) in info_by_id for t in tasks), f"Annotation không thuộc split {EVAL_SPLIT}."
assert all(t.get("Polygons") and t.get("phrase", "").strip() for t in tasks), "Thiếu polygon GT hoặc phrase."
assert all(re.fullmatch(r"[A-Za-z0-9_-]+", str(t["task_id"])) for t in tasks), "Task ID không hợp lệ."

selected = sorted(tasks, key=lambda t: str(t["task_id"]))
if MAX_QUERIES is not None and MAX_QUERIES < len(selected):
    selected = random.Random(SEED).sample(selected, MAX_QUERIES)
selected.sort(key=lambda t: (int(t["image_id"]), str(t["task_id"])))
image_ids = sorted({int(t["image_id"]) for t in selected})
if len(image_ids) < 2:
    raise ValueError("Cần ít nhất 2 image_id để chia calibration/evaluation; tăng MAX_QUERIES.")
shuffled_ids = image_ids.copy()
random.Random(SEED).shuffle(shuffled_ids)
n_cal = min(len(image_ids) - 1, max(1, round(len(image_ids) * CALIBRATION_FRACTION)))
calibration_ids = set(shuffled_ids[:n_cal])
group_for_image = {i: "calibration" if i in calibration_ids else "evaluation" for i in image_ids}
assert not calibration_ids.intersection(set(image_ids) - calibration_ids)
write_json(OUT / "selected_tasks.json", [{"task_id": str(t["task_id"]), "image_id": int(t["image_id"]),
    "phrase": t["phrase"], "group": group_for_image[int(t["image_id"])]} for t in selected])
print(f"Annotation {EVAL_SPLIT}: {len(tasks)} queries / {len(info_by_id)} images")
print(f"Selected: {len(selected)} queries / {len(image_ids)} images; calibration {n_cal} images")


## 4. Tải và kiểm tra ảnh

Ưu tiên ảnh đã có trong dataset input. Ảnh thiếu sẽ tải từ URL Visual Genome trong metadata
bằng HTTPS vào working cache, retry có giới hạn. Lỗi tải hoặc kích thước khác metadata làm
dừng lượt chạy: không âm thầm bỏ query để tạo số liệu đẹp hơn.


In [ ]:
image_cache = WORK_ROOT / "phrasecut_image_cache"
image_cache.mkdir(parents=True, exist_ok=True)
def image_search_roots(*extra):
    """Nơi tìm ảnh <image_id>.jpg: IMAGE_ROOT nếu đặt, nếu không thì mọi Kaggle Input; cộng thư mục annotation."""
    roots = [Path(IMAGE_ROOT)] if IMAGE_ROOT else ([Path("/kaggle/input")] if Path("/kaggle/input").exists() else [])
    roots += [Path(r) for r in extra]
    # Bỏ thư mục nằm trong thư mục đã quét để không quét hai lần.
    return [r for i, r in enumerate(roots) if r.is_dir() and not any(r == q or q in r.parents for q in roots[:i])]

def index_local_images(roots, wanted_ids):
    index = {}
    for search_root in roots:
        for path in sorted(search_root.rglob("*")):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"} and path.stem.isdigit() and int(path.stem) in wanted_ids:
                index.setdefault(int(path.stem), path)
    return index

local_images = index_local_images(image_search_roots(data_root), set(image_ids))
print(f"Ảnh có sẵn trong Input: {len(local_images)}/{len(image_ids)}; phần còn lại tải từ Visual Genome")

def valid_image(path, info):
    with Image.open(path) as image:
        image.load()
        if image.size != (int(info["width"]), int(info["height"])):
            raise ValueError(f"{path.name}: ảnh {image.size} != metadata {(info['width'], info['height'])}")

def prepare_image(image_id):
    info = info_by_id[image_id]
    path = local_images.get(image_id, image_cache / f"{image_id}.jpg")
    if path.is_file():
        valid_image(path, info)
        return image_id, path
    url = info["url"].replace("http://", "https://", 1)
    temp = path.with_suffix(".download")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(10, 60)) as response:
                response.raise_for_status()
                with temp.open("wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        f.write(chunk)
            valid_image(temp, info)
            temp.replace(path)
            return image_id, path
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)

image_paths, image_errors = {}, []
def prepare_safe(image_id):
    try:
        return prepare_image(image_id), None
    except Exception as exc:
        return None, {"image_id": image_id, "error": str(exc)}

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    for result, error in tqdm(pool.map(prepare_safe, image_ids), total=len(image_ids), desc="Images"):
        if error:
            image_errors.append(error)
        else:
            image_paths[result[0]] = result[1]
write_json(OUT / "image_errors.json", image_errors)
if image_errors:
    raise RuntimeError(f"{len(image_errors)} ảnh lỗi; xem image_errors.json. Sửa dữ liệu rồi chạy lại.")
print("Validated images:", len(image_paths))


## 5. Nạp model và định nghĩa inference/metric

Dùng [triển khai CLIPSeg của Transformers](https://huggingface.co/docs/transformers/v4.55.4/en/model_doc/clipseg),
revision checkpoint được cố định. Tiền xử lý, rasterize GT và metric bám theo code đánh giá của tác giả
([score.py](https://github.com/timojl/clipseg/blob/master/score.py),
[metrics.py](https://github.com/timojl/clipseg/blob/master/metrics.py) `FixedIntervalMetrics`,
[datasets/phrasecut.py](https://github.com/timojl/clipseg/blob/master/datasets/phrasecut.py)):

- Ảnh resize **352×352** bằng bilinear `align_corners=True`, chia 255, chuẩn hóa mean/std; text giữ nguyên phrase.
- GT: hợp polygon bằng `skimage.draw.polygon2mask`, rồi resize nearest về 352×352. Chấm **ở 352×352**, sigmoid, FP32.
- 49 ngưỡng 0,02–0,98, pixel dương khi `p > t`. Với từng ngưỡng: `miou` = trung bình theo query của
  `tp/(1+tp+fp+fn)`; `fgiou` = cộng dồn mọi pixel; AP = tích phân Simpson của đường precision–recall cộng dồn.
- Bảng paper dùng `miou_0.3`, `fgiou_0.3`, `ap` (cột `pc_miou_0.3`, `pc_fgiou_0.3`, `pc_ap` trong `experiments/phrasecut.yaml`).

Bên cạnh đó vẫn giữ metric ở độ phân giải ảnh gốc (IoU theo query, cumulative IoU, AP pixel theo query)
để so với các lượt chạy trước. Không dùng GT để tạo text prompt hoặc thay đổi đầu vào ảnh.


In [ ]:
load_kwargs = {"local_files_only": True} if MODEL_PATH else {"revision": MODEL_REVISION}
model_source = str(MODEL_PATH) if MODEL_PATH else MODEL_ID
processor = CLIPSegProcessor.from_pretrained(model_source, use_fast=False, **load_kwargs)
model = CLIPSegForImageSegmentation.from_pretrained(model_source, **load_kwargs).to(device).eval()
assert processor.image_processor.size == {"height": 352, "width": 352}, "Processor không có input 352 như baseline."
assert model.config.vision_config.patch_size == 16, "Baseline yêu cầu ViT-B/16."
print("Model:", model_source, "| parameters:", sum(p.numel() for p in model.parameters()))

try:
    from skimage.draw import polygon2mask
except ImportError as exc:
    raise ImportError("Cần scikit-image (có sẵn trên Kaggle); cài bằng pip install scikit-image.") from exc
try:
    from scipy.integrate import simpson
except ImportError:  # SciPy cũ
    from scipy.integrate import simps as simpson

# Tiền xử lý và độ phân giải chấm điểm theo score.py + datasets/phrasecut.py của CLIPSeg.
PAPER_SIZE = 352
PAPER_THRESHOLDS = np.linspace(0, 1, 51)[1:-1]   # 49 ngưỡng 0,02–0,98 như FixedIntervalMetrics
image_mean = torch.tensor(processor.image_processor.image_mean, dtype=torch.float32).view(3, 1, 1)
image_std = torch.tensor(processor.image_processor.image_std, dtype=torch.float32).view(3, 1, 1)

def rasterize_gt(task, width, height, allow_empty=False):
    """Hợp polygon mọi instance bằng skimage polygon2mask (x, y đảo thành hàng, cột), như PhraseCut.load_sample."""
    target = np.zeros((height, width), dtype=bool)
    for instance in task["Polygons"]:
        for polygon in instance:
            if len(polygon) >= 3:
                target |= polygon2mask((height, width), np.asarray(polygon, dtype=np.float64)[:, ::-1])
    if not allow_empty and not target.any():
        raise ValueError(f"GT rỗng: task {task['task_id']}")
    return target

def paper_pixel_values(image):
    """Ảnh RGB → [3,352,352]: resize bilinear align_corners=True trên ảnh gốc, /255, chuẩn hóa mean/std."""
    pixels = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1).unsqueeze(0)
    pixels = F.interpolate(pixels, size=(PAPER_SIZE, PAPER_SIZE), mode="bilinear", align_corners=True)[0]
    return (pixels / 255 - image_mean) / image_std

def paper_target(target):
    """GT ở độ phân giải gốc → 352×352 bằng nearest, như nnf.interpolate trong load_sample."""
    mask = torch.from_numpy(target.astype(np.uint8))[None, None]
    return F.interpolate(mask, size=(PAPER_SIZE, PAPER_SIZE), mode="nearest")[0, 0].numpy().astype(bool)

def paper_threshold_stats(probability, target):
    """tp, fp, fn, tn tại 49 ngưỡng (dương khi p > t) cho một query, như FixedIntervalMetrics.add."""
    order = np.argsort(probability, axis=None, kind="stable")
    p, g = probability.ravel()[order], target.ravel()[order]
    below = np.searchsorted(p, PAPER_THRESHOLDS, side="right")   # số pixel p <= t
    fn = np.concatenate([[0], np.cumsum(g, dtype=np.int64)])[below]
    tp = int(g.sum()) - fn
    fp = (len(p) - below) - tp
    tn = below - fn
    return np.stack([tp, fp, fn, tn]).astype(np.int64)

def paper_metrics(stats):
    """Metric bảng PhraseCut của paper từ stats [Q,4,49]: miou (trung bình theo query), fgiou (cộng dồn), AP Simpson."""
    stats = np.asarray(stats, dtype=np.float64)
    tp, fp, fn = stats[:, 0], stats[:, 1], stats[:, 2]
    s_tp, s_fp, s_fn = tp.sum(0), fp.sum(0), fn.sum(0)
    made = (s_tp + s_fp) > 0
    pairs = sorted(set(zip((s_tp / (1 + s_tp + s_fn))[made], (s_tp / (1 + s_tp + s_fp))[made])))
    recalls, precisions = (np.array(v) for v in zip(*pairs))
    fgiou = s_tp / (1 + s_tp + s_fp + s_fn)
    miou = (tp / (1 + tp + fp + fn)).mean(0)
    index = {t: int(np.argmin(np.abs(PAPER_THRESHOLDS - t))) for t in (0.3, 0.5)}
    return {"n_queries": len(stats), "miou_0.3": float(miou[index[0.3]]), "fgiou_0.3": float(fgiou[index[0.3]]),
            "fgiou_0.5": float(fgiou[index[0.5]]), "ap": float(simpson(precisions, x=recalls)),
            "miou_best": float(miou.max()), "miou_best_t": float(PAPER_THRESHOLDS[miou.argmax()]),
            "fgiou_best": float(fgiou.max()), "fgiou_best_t": float(PAPER_THRESHOLDS[fgiou.argmax()])}

def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)

@torch.inference_mode()
def predict_batch(image, batch_tasks, with_paper=False):
    """Xác suất ở kích thước ảnh gốc; with_paper=True trả thêm xác suất 352×352 để chấm theo paper."""
    prompts = [PROMPT_TEMPLATE.format(phrase=t["phrase"]) for t in batch_tasks]
    inputs = dict(processor.tokenizer(text=prompts, padding=True, truncation=True, return_tensors="pt"))
    inputs["pixel_values"] = paper_pixel_values(image).unsqueeze(0).repeat(len(prompts), 1, 1, 1)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    synchronize()
    started = time.perf_counter()
    context = torch.autocast(device_type="cuda", dtype=torch.float16) if EVAL_AMP else nullcontext()
    with context:
        logits = segment(model, inputs["pixel_values"], inputs["input_ids"], inputs["attention_mask"])
    logits = logits.float().reshape(len(prompts), *logits.shape[-2:])
    probabilities = F.interpolate(logits.unsqueeze(1), size=(image.height, image.width),
        mode="bilinear", align_corners=False).squeeze(1).sigmoid()
    synchronize()
    seconds = time.perf_counter() - started
    values = probabilities.cpu().numpy()
    if not np.isfinite(values).all():
        raise ValueError("Prediction chứa NaN/Inf; thử EVAL_AMP=False.")
    if with_paper:
        return values, logits.sigmoid().cpu().numpy(), seconds
    return values, seconds

def binary_metrics(probability, target, threshold):
    if probability.shape != target.shape or not target.any():
        raise ValueError("Shape không khớp hoặc GT rỗng.")
    prediction = probability >= threshold
    intersection = int(np.logical_and(prediction, target).sum())
    union = int(np.logical_or(prediction, target).sum())
    return {"intersection": intersection, "union": union, "iou": intersection / union,
            "dice": 2 * intersection / int(prediction.sum() + target.sum())}

def summarize(rows):
    if not rows:
        raise ValueError("Không có query để tính metric.")
    result = {"n_queries": len(rows), "n_images": len({r["image_id"] for r in rows}),
        "mean_sample_iou": float(np.mean([r["iou"] for r in rows])),
        "cumulative_iou": sum(r["intersection"] for r in rows) / sum(r["union"] for r in rows),
        "mean_dice": float(np.mean([r["dice"] for r in rows]))}
    for cutoff in [0.3, 0.5, 0.7, 0.9]:
        result[f"precision_at_{cutoff}"] = float(np.mean([r["iou"] >= cutoff for r in rows]))
    if "pixel_ap" in rows[0]:
        result["mean_sample_pixel_ap"] = float(np.mean([r["pixel_ap"] for r in rows]))
    return result


## 5b. LoRA cho CLIP vision và forward dùng chung

`segment` thay cho `model(...)`: tính activation CLIP, conditional embedding (text, hoặc trộn text–ảnh khi train
PhraseCut+), rồi gọi decoder. Với model gốc, `segment` cho đúng logits của `CLIPSegForImageSegmentation.forward`
(kiểm tra ngay bên dưới), nên B1 và E1 dùng cùng một phép tính. Khi có LoRA, activation được tính lại với gradient
(chỉ tới layer cuối decoder dùng), còn embedding điều kiện vẫn tính bằng CLIP gốc.


In [ ]:
import math
from contextlib import contextmanager
import torch.utils.checkpoint

class LoRALinear(torch.nn.Module):
    """W·x + (alpha/r)·B·A·x; B khởi tạo 0 nên lúc đầu đúng bằng lớp gốc. Lớp gốc giữ đóng băng."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lora_a = torch.nn.Parameter(torch.empty(rank, base.in_features, device=base.weight.device))
        self.lora_b = torch.nn.Parameter(torch.zeros(base.out_features, rank, device=base.weight.device))
        torch.nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))
        self.scale = alpha / rank
        self.enabled = True

    def forward(self, x):
        out = self.base(x)
        if self.enabled:
            out = out + (x @ self.lora_a.t().to(x.dtype)) @ self.lora_b.t().to(x.dtype) * self.scale
        return out

def lora_layers(model):
    """Các layer CLIP vision có output đi vào decoder (0..max(extract_layers))."""
    return list(model.clip.vision_model.encoder.layers[:max(model.extract_layers) + 1])

def lora_modules(model):
    return [m for m in model.clip.vision_model.modules() if isinstance(m, LoRALinear)]

def has_lora(model):
    return bool(lora_modules(model))

def attach_lora(model, rank, alpha):
    assert not has_lora(model), "LoRA đã được gắn"
    for layer in lora_layers(model):
        for name in ["q_proj", "v_proj"]:
            setattr(layer.self_attn, name, LoRALinear(getattr(layer.self_attn, name), rank, alpha))

def lora_parameters(model):
    return [p for m in lora_modules(model) for p in (m.lora_a, m.lora_b)]

def trainable_groups(model, lr):
    """Nhóm 0 = decoder (lr), nhóm 1 = LoRA (lr × LORA_LR_SCALE). Không có LoRA thì chỉ một nhóm, như B1."""
    groups = [{"params": list(model.decoder.parameters()), "lr": lr}]
    if lora_parameters(model):
        groups.append({"params": lora_parameters(model), "lr": lr * LORA_LR_SCALE})
    return groups

def sync_lora_lr(optimizer):
    """Giữ LR của LoRA đúng bằng LORA_LR_SCALE × LR decoder sau mỗi bước của lịch cosine."""
    if len(optimizer.param_groups) > 1:
        optimizer.param_groups[1]["lr"] = optimizer.param_groups[0]["lr"] * LORA_LR_SCALE

def lora_state_dict(model):
    return {k: v.detach().cpu() for k, v in model.clip.vision_model.state_dict().items() if "lora_" in k}

def load_lora_state(model, state):
    missing, unexpected = model.clip.vision_model.load_state_dict(state, strict=False)
    assert not unexpected and not [k for k in missing if "lora_" in k], (missing, unexpected)

@contextmanager
def lora_disabled(model):
    """Embedding điều kiện (text, ảnh support) luôn tính bằng CLIP gốc, kể cả ở E4c."""
    modules = lora_modules(model)
    for m in modules:
        m.enabled = False
    try:
        yield
    finally:
        for m in modules:
            m.enabled = True

def vision_activations(model, pixel_values):
    """Output các layer extract_layers của CLIP vision, chỉ chạy tới layer cuối cần dùng."""
    vision = model.clip.vision_model
    hidden = vision.pre_layrnorm(vision.embeddings(pixel_values, interpolate_pos_encoding=True))
    activations = []
    for index, layer in enumerate(lora_layers(model)):
        if LORA_GRAD_CHECKPOINT and torch.is_grad_enabled():
            hidden = torch.utils.checkpoint.checkpoint(layer, hidden, None, None, use_reentrant=False)[0]
        else:
            hidden = layer(hidden, None, None)[0]
        if index in model.extract_layers:
            activations.append(hidden)
    return activations

def mix_conditionals(text_cond, visual_cond, text_weights):
    """cond = w·text + (1−w)·ảnh (giống cell PhraseCut+; định nghĩa sớm để segment dùng được khi chỉ đánh giá)."""
    weights = text_weights.to(text_cond.device, text_cond.dtype)
    return text_cond * weights + visual_cond * (1 - weights)

def segment(model, pixel_values, input_ids, attention_mask, support_pixel_values=None, text_weights=None):
    """Logits [B,352,352] cho train lẫn inference, model gốc (B1) hoặc CLIP vision có LoRA (E4c)."""
    size = pixel_values.shape[0]
    lora = has_lora(model)
    with torch.no_grad():
        if not lora:
            hidden = model.clip.vision_model(pixel_values=pixel_values, output_hidden_states=True,
                                             interpolate_pos_encoding=True).hidden_states
            activations = [hidden[i + 1] for i in model.extract_layers]
        with lora_disabled(model):
            cond = model.get_conditional_embeddings(batch_size=size, input_ids=input_ids, attention_mask=attention_mask)
            if support_pixel_values is not None:
                visual = model.get_conditional_embeddings(batch_size=size, conditional_pixel_values=support_pixel_values)
                cond = mix_conditionals(cond, visual, text_weights)
    if lora:
        activations = vision_activations(model, pixel_values)  # gradient chỉ tới tham số LoRA
    logits = model.decoder(activations, cond).logits
    return logits.reshape(size, *logits.shape[-2:])

def experiment_logits(model, pixel_values, input_ids=None, attention_mask=None, conditional_pixel_values=None):
    """E2/E3 forward: text hoặc ảnh support; với LoRA, embedding điều kiện vẫn dùng CLIP gốc."""
    if not has_lora(model):
        return model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask,
                     conditional_pixel_values=conditional_pixel_values).logits
    if (input_ids is None) == (conditional_pixel_values is None):
        raise ValueError("Supply text or support images, not both")
    if input_ids is not None:
        return segment(model, pixel_values, input_ids, attention_mask)
    with torch.no_grad():
        activations = vision_activations(model, pixel_values)
        with lora_disabled(model):
            cond = model.get_conditional_embeddings(batch_size=pixel_values.shape[0],
                                                    conditional_pixel_values=conditional_pixel_values)
    return model.decoder(activations, cond).logits

# segment với model gốc phải trùng forward của Transformers.
with torch.no_grad():
    check = processor.tokenizer(["a red car", "dog"], padding=True, return_tensors="pt").to(device)
    check_pixels = torch.randn(2, 3, PAPER_SIZE, PAPER_SIZE, device=device)
    reference = model(pixel_values=check_pixels, **check).logits
    assert torch.allclose(segment(model, check_pixels, check["input_ids"], check["attention_mask"]),
                          reference, atol=1e-4), "segment lệch forward của Transformers"
    hidden = model.clip.vision_model(pixel_values=check_pixels, output_hidden_states=True).hidden_states
    for ours, theirs in zip(vision_activations(model, check_pixels), [hidden[i + 1] for i in model.extract_layers]):
        assert torch.allclose(ours, theirs, atol=1e-4), "vision_activations lệch hidden_states của Transformers"
print("segment khớp CLIPSegForImageSegmentation.forward")


In [ ]:
"""Preserve every training sample; diagnose AMP failures and retry in FP32.

Embedded in the Kaggle notebook by build_e4_notebook.py. No model is run here.
"""
import json
from pathlib import Path

import torch
import torch.nn.functional as F


NUMERICAL_POLICY = "amp_with_verified_fp32_batch_retry_v1"


def tensor_summary(value):
    values = []
    if isinstance(value, torch.Tensor) and value.is_floating_point():
        finite = torch.isfinite(value)
        values.append({"shape": list(value.shape), "dtype": str(value.dtype),
                       "finite": bool(finite.all()),
                       "nonfinite": int((~finite).sum()),
                       "finite_abs_max": float(value.detach()[finite].abs().max()) if finite.any() else None})
    elif isinstance(value, (tuple, list, dict)):
        for item in value.values() if isinstance(value, dict) else value:
            values.extend(tensor_summary(item))
    return values


def save_numerical_report(directory, report):
    path = Path(directory) / "numerical_recovery.json"
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(report, indent=2), encoding="utf-8")
    temporary.replace(path)


def loss_backward_with_recovery(forward, model, inputs, labels, scaler, directory,
                                epoch, batch_index, global_step, microbatch_size=16):
    """One effective batch and one optimizer step, including a rare FP32 retry.

    The caller owns optimizer.zero_grad/step and the scheduler. FP32 microbatch
    losses are weighted by sample count before backward, so batch 64 stays 64.
    A nonfinite FP32 loss raises before any optimizer step. No batch is dropped.
    """
    logits = forward(**inputs)
    loss = F.binary_cross_entropy_with_logits(logits.float(), labels.to(logits.device))
    if torch.isfinite(loss):
        scaler.scale(loss).backward()
        return loss.detach()

    # Release the invalid graph before the diagnostic replay and FP32 backward.
    logits, loss = None, None
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / "numerical_recovery.json"
    report = json.loads(report_path.read_text(encoding="utf-8")) if report_path.exists() else {
        "policy": NUMERICAL_POLICY, "events": []}
    event = {"epoch": epoch, "batch_index": batch_index, "global_step": global_step,
             "effective_batch_size": len(labels), "fp32_microbatch_size": microbatch_size,
             "inputs": {key: tensor_summary(value) for key, value in inputs.items()},
             "labels": tensor_summary(labels), "first_nonfinite_modules": []}
    report["events"].append(event)
    # Always retain the first failing batch and its model weights for GPU replay.
    if len(report["events"]) == 1:
        torch.save({"inputs": {k: v.detach().cpu() for k, v in inputs.items()},
                    "labels": labels.detach().cpu(),
                    "trainable_state": {k: p.detach().cpu() for k, p in model.named_parameters() if p.requires_grad},
                    "epoch": epoch, "batch_index": batch_index, "global_step": global_step},
                   directory / "first_nonfinite_batch.pt")
    tensors = [t for t in inputs.values() if t.is_floating_point()] + [labels]
    if not all(torch.isfinite(t).all() for t in tensors):
        event["classification"] = "nonfinite_input_or_label"
        save_numerical_report(directory, report)
        raise ValueError("Nonfinite training input/label; diagnostic saved, no update")
    if not all(torch.isfinite(p).all() for p in model.parameters()):
        event["classification"] = "nonfinite_model_parameters"
        save_numerical_report(directory, report)
        raise ValueError("Nonfinite model parameters; diagnostic saved, no update")

    def hook(name):
        def inspect(module, args, output):
            # First failing leaf/compound on each device. All hooks are removed
            # before backward so a graph never retains diagnostic tensors.
            if len(event["first_nonfinite_modules"]) < 6:
                summaries = tensor_summary(output)
                if any(not item["finite"] for item in summaries):
                    event["first_nonfinite_modules"].append({"name": name,
                        "type": type(module).__name__, "inputs": tensor_summary(args), "outputs": summaries})
        return inspect

    handles = [module.register_forward_hook(hook(name)) for name, module in model.named_modules()]
    try:
        with torch.no_grad():
            replay = forward(**inputs)
            event["amp_replay_finite"] = bool(torch.isfinite(replay).all())
            del replay
    finally:
        for handle in handles:
            handle.remove()
    event["classification"] = "awaiting_fp32_same_batch_check"
    save_numerical_report(directory, report)
    print("AMP_NONFINITE_DIAGNOSTIC " + json.dumps(event), flush=True)

    total, result_device = 0.0, labels.device
    try:
        for start in range(0, len(labels), microbatch_size):
            end = min(start + microbatch_size, len(labels))
            chunk = {key: value[start:end] for key, value in inputs.items()}
            logits = forward(**chunk, force_fp32=True)
            result_device = logits.device
            part = F.binary_cross_entropy_with_logits(logits.float(), labels[start:end].to(logits.device))
            if not torch.isfinite(part):
                raise ValueError("Same batch is nonfinite in FP32 too; no optimizer update")
            weight = (end - start) / len(labels)
            scaler.scale(part * weight).backward()
            total += float(part.detach()) * weight
            del logits, part
    except Exception:
        model.zero_grad(set_to_none=True)
        event["classification"] = "fp32_retry_failed"
        save_numerical_report(directory, report)
        raise
    event.update(classification="fp16_numerical_failure_fp32_same_batch_finite", fp32_loss=total,
                 backward_completed=True, samples_skipped=0)
    save_numerical_report(directory, report)
    print("AMP_RECOVERED " + json.dumps(event), flush=True)
    return torch.tensor(total, device=result_device)

write_json(OUT / "numerical_policy.json", {"policy": NUMERICAL_POLICY, "effective_batch_size": BATCH_SIZE, "fp32_retry_microbatch_size": 16, "on_nonfinite_fp32": "stop_before_optimizer_update", "samples_skipped": 0})
print("Numerical policy:", NUMERICAL_POLICY, flush=True)


## 6. Biến PhraseCut train thành PhraseCut+ và tính số bước từ epoch

PhraseCut+ không phải file dữ liệu riêng: nó được tạo **ngẫu nhiên mỗi lần lấy mẫu** từ PhraseCut,
theo `PhraseCutPlus` trong [datasets/phrasecut.py](https://github.com/timojl/clipseg/blob/master/datasets/phrasecut.py)
và vòng train trong [training.py](https://github.com/timojl/clipseg/blob/master/training.py) của tác giả:

1. **Crop vuông ngẫu nhiên** cạnh min(H, W), ưu tiên crop chứa vật (`find_crop`), resize 352.
2. **Mẫu âm** với xác suất `NEGATIVE_PROB`: thay cụm từ bằng cụm khác, mask đáp án rỗng.
3. **Support** (`TRAIN_MODE="phrasecut_plus"`): chọn ngẫu nhiên một mẫu train **ở ảnh khác** có cùng cụm từ,
   dùng mask GT của chính ảnh support để làm mờ nền, giữ nền 10%, crop quanh vật (nới 10%), pad vuông, resize 352
   (`crop_blur_highlight352`). Cụm từ không xuất hiện ở ảnh khác thì không có support.
4. **Template** ngẫu nhiên trong 9 template `shuffle+`.
5. Trong vòng train: `cond = w·text + (1−w)·ảnh`, `w ~ U(0, MIX_TEXT_MAX)`; mẫu không có support dùng `w = 1`.

`TRAIN_MODE="text_only"` bỏ bước 3 và 5, tương ứng cấu hình `rd64-uni-novis` của tác giả (ablation chỉ học text).
Khác bản gốc: support luôn lấy từ ảnh khác (bản gốc có thể chọn trúng chính mẫu đó). GT rasterize bằng skimage như bản gốc.

`steps_per_epoch = ceil(N_train / BATCH_SIZE)`, `total_steps = EPOCHS × steps_per_epoch`.
Mỗi epoch duyệt hết tập train đã chọn một lần, shuffle, **không bỏ batch cuối**.
Tác giả train batch 64, 20.000 bước (`rd64-uni`) hoặc 50.000 bước (`rd64-uni-refined`), AMP;
với 310.816 query train, 20.000 × 64 tương đương khoảng 4 epoch.
Mặc định reset decoder ngẫu nhiên, giữ CLIP pretrained; `TRAIN_INIT="pretrained_decoder"`
fine-tune decoder của checkpoint refined.

Train full cần annotation ~344 MB và nhiều GB ảnh; nên Add Input dataset đã cache.
Nếu `TRAIN_ROOT=None`, tự tìm `refer_train.json` trong Input; nếu chưa có thì tải annotation
chính thức và tải ảnh train theo nhu cầu vào working cache. `MAX_TRAIN_QUERIES=8` để smoke.
Miniv chỉ dùng đánh giá sau train, không dùng cập nhật trọng số hoặc chọn checkpoint.


In [ ]:
from torch.utils.data import Dataset, DataLoader

# 9 template "shuffle+" trong models/clipseg.py của repo CLIPSeg gốc.
SHUFFLE_PLUS_TEMPLATES = ["a photo of a {}.", "a photograph of a {}.", "an image of a {}.", "{}.",
    "a cropped photo of a {}.", "a good photo of a {}.", "a photo of one {}.", "a bad photo of a {}.",
    "a photo of the {}."]

def random_square_crop(target, iterations=50, min_frac=0.05):
    """Crop vuông cạnh min(H, W); nhận crop đầu tiên có số pixel vật > min_frac × diện tích ảnh (find_crop gốc)."""
    height, width = target.shape
    side = min(height, width)
    min_sum = height * width * min_frac
    best = (-1, 0, 0)
    for _ in range(iterations):
        y = int(torch.randint(0, height - side + 1, (1,)))
        x = int(torch.randint(0, width - side + 1, (1,)))
        total = int(target[y:y + side, x:x + side].sum())
        if total > min_sum:
            return y, x, side
        if total > best[0]:
            best = (total, y, x)
    return best[1], best[2], side

def gaussian_blur(image, kernel_size=15, sigma=3.0):
    """Gaussian blur [C,H,W] biên reflect, tương đương cv2.GaussianBlur(img, (15, 15), 3) của repo gốc."""
    coords = torch.arange(kernel_size, dtype=image.dtype) - (kernel_size - 1) / 2
    kernel = torch.exp(-coords ** 2 / (2 * sigma ** 2))
    kernel = kernel / kernel.sum()
    channels, pad = image.shape[0], kernel_size // 2
    blurred = F.pad(image.unsqueeze(0), (pad, pad, pad, pad), mode="reflect")
    blurred = F.conv2d(blurred, kernel.view(1, 1, 1, -1).repeat(channels, 1, 1, 1), groups=channels)
    blurred = F.conv2d(blurred, kernel.view(1, 1, -1, 1).repeat(channels, 1, 1, 1), groups=channels)
    return blurred.squeeze(0)

def visual_prompt(image, mask, size=352, blur_sigma=3.0, background_factor=0.1, context=0.1):
    """crop_blur_highlight352: nền làm mờ còn 10% (trên ảnh đã chuẩn hóa), crop quanh vật nới 10%, pad vuông, resize."""
    mask = mask.to(image.dtype)
    blended = image * mask + background_factor * gaussian_blur(image, sigma=blur_sigma) * (1 - mask)
    ys, xs = torch.nonzero(mask > 0, as_tuple=True)
    if len(ys):
        y0, y1, x0, x1 = int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1
    else:
        y0, y1, x0, x1 = 0, mask.shape[0], 0, mask.shape[1]
    dy, dx = context * (y1 - y0), context * (x1 - x0)
    y0, y1 = int(max(0, y0 - dy)), int(min(mask.shape[0], y1 + dy))
    x0, x1 = int(max(0, x0 - dx)), int(min(mask.shape[1], x1 + dx))
    crop = blended[:, y0:y1, x0:x1]
    height, width = crop.shape[1:]
    side = max(height, width)
    top, left = (side - height) // 2, (side - width) // 2
    crop = F.pad(crop, (left, side - width - left, top, side - height - top))
    return F.interpolate(crop.unsqueeze(0), size=(size, size), mode="nearest").squeeze(0)

def sample_text_weights(support_valid, max_text_weight):
    """w ~ U(0, max) theo từng mẫu; mẫu không có support dùng w = 1. Lấy mẫu trên CPU trước khi chia GPU."""
    weights = torch.rand(len(support_valid), 1) * max_text_weight
    return torch.maximum(weights, (~support_valid.cpu()).float()[:, None])

def mix_conditionals(text_cond, visual_cond, text_weights):
    """cond = w·text + (1−w)·ảnh."""
    weights = text_weights.to(text_cond.device, text_cond.dtype)
    return text_cond * weights + visual_cond * (1 - weights)

training_metadata = {"enabled": TRAIN_DECODER}
if TRAIN_DECODER:
    if TRAIN_ROOT is None:
        train_candidates = sorted(Path("/kaggle/input").rglob("refer_train.json")) if Path("/kaggle/input").exists() else []
        if len(train_candidates) > 1:
            raise ValueError("Có nhiều dataset train; đặt TRAIN_ROOT rõ ràng.")
        train_root = train_candidates[0].parent if train_candidates else WORK_ROOT / "phrasecut_train"
    else:
        train_root = Path(TRAIN_ROOT)
        if not train_root.is_dir():
            raise FileNotFoundError(f"TRAIN_ROOT không tồn tại: {train_root}")
    train_root.mkdir(parents=True, exist_ok=True)
    train_annotation_path = train_root / "refer_train.json"
    train_raw = ensure_json(train_annotation_path, REFER_DRIVE_IDS["train"])
    train_info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == "train"}
    # Các image_id sai kích thước được loại trong wrapper CLIPSeg gốc.
    bad_image_ids = {150417, 285665, 498246, 61564, 285743, 498269, 498010, 150516,
                     150344, 286093, 61530, 150333, 286065, 285814, 498187, 285761, 498042}
    train_tasks = []
    excluded_train = []
    for task in train_raw:
        image_id = int(task["image_id"])
        if image_id not in train_info_by_id:
            raise ValueError(f"Task {task['task_id']} không thuộc split train chính thức.")
        if image_id in bad_image_ids:
            excluded_train.append({"task_id": str(task["task_id"]), "image_id": image_id,
                                   "reason": "known_dimension_mismatch_in_original_clipseg_wrapper"})
            continue
        if not task.get("Polygons") or not task.get("phrase", "").strip():
            raise ValueError(f"Train task thiếu GT hoặc phrase: {task['task_id']}")
        train_tasks.append({key: task[key] for key in ["task_id", "image_id", "phrase", "Polygons"]})
    del train_raw
    train_tasks.sort(key=lambda t: str(t["task_id"]))
    full_train_queries = len(train_tasks)
    if MAX_TRAIN_QUERIES is not None and MAX_TRAIN_QUERIES < len(train_tasks):
        train_tasks = random.Random(SEED).sample(train_tasks, MAX_TRAIN_QUERIES)
    if not train_tasks:
        raise ValueError("Train dataset rỗng.")
    train_image_ids = {int(t["image_id"]) for t in train_tasks}
    assert train_image_ids.isdisjoint(info_by_id), f"Train/{EVAL_SPLIT} bị trùng image_id."
    assert len({str(t["task_id"]) for t in train_tasks}) == len(train_tasks), "Train task ID bị trùng."
    write_json(OUT / "excluded_train_tasks.json", excluded_train)
    write_json(OUT / "train_tasks.json", [{"task_id": str(t["task_id"]), "image_id": int(t["image_id"]),
                                          "phrase": t["phrase"]} for t in train_tasks])
    train_local_images = index_local_images(image_search_roots(train_root), train_image_ids)
    print(f"Ảnh train có sẵn trong Input: {len(train_local_images)}/{len(train_image_ids)}; "
          "phần còn lại tải từ Visual Genome trong lúc train")
    train_image_cache = WORK_ROOT / "phrasecut_train_image_cache"
    train_image_cache.mkdir(parents=True, exist_ok=True)

    def get_train_image_path(image_id):
        path = train_local_images.get(image_id, train_image_cache / f"{image_id}.jpg")
        info = train_info_by_id[image_id]
        if path.is_file():
            return path
        # Temp riêng cho từng worker, tránh ghi đè nếu nhiều query có cùng ảnh.
        temp = path.with_name(f"{image_id}.{os.getpid()}.download")
        url = info["url"].replace("http://", "https://", 1)
        for attempt in range(3):
            try:
                with requests.get(url, stream=True, timeout=(10, 60)) as response:
                    response.raise_for_status()
                    with temp.open("wb") as f:
                        for chunk in response.iter_content(1024 * 1024):
                            f.write(chunk)
                valid_image(temp, info)
                temp.replace(path)
                return path
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(attempt + 1)

    # Chỉ mục cụm từ -> các mẫu train, dùng để chọn support và mẫu âm.
    tasks_by_phrase = defaultdict(list)
    for index, task in enumerate(train_tasks):
        tasks_by_phrase[task["phrase"]].append(index)
    all_train_phrases = sorted(tasks_by_phrase)
    queries_with_support = sum(1 for task in train_tasks if any(
        train_tasks[i]["image_id"] != task["image_id"] for i in tasks_by_phrase[task["phrase"]]))

    def load_train_sample(task):
        """Ảnh chuẩn hóa [3,352,352] và mask [352,352], crop vuông ngẫu nhiên như load_sample gốc."""
        image_id = int(task["image_id"])
        with Image.open(get_train_image_path(image_id)) as raw:
            image = raw.convert("RGB")
        info = train_info_by_id[image_id]
        if image.size != (int(info["width"]), int(info["height"])):
            raise ValueError(f"Train ảnh {image_id} sai kích thước metadata.")
        # Như bản gốc: polygon quá mảnh có thể cho GT rỗng; mẫu đó vẫn được train với mask rỗng.
        target = rasterize_gt(task, image.width, image.height, allow_empty=True)
        pixels = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1)
        if AUG_CROP:
            y, x, side = random_square_crop(target)
            target, pixels = target[y:y + side, x:x + side], pixels[:, y:y + side, x:x + side]
        pixels = F.interpolate(pixels.unsqueeze(0), size=(352, 352), mode="bilinear", align_corners=True).squeeze(0)
        mask = F.interpolate(torch.from_numpy(target.astype(np.float32))[None, None], size=(352, 352), mode="nearest")[0, 0]
        return (pixels / 255 - image_mean) / image_std, mask

    class PhraseCutPlusDataset(Dataset):
        """PhraseCut+ tạo lại ở mỗi lần lấy mẫu: mẫu âm, support cùng cụm từ từ ảnh khác, template ngẫu nhiên."""
        def __len__(self):
            return len(train_tasks)

        def __getitem__(self, index):
            task = train_tasks[index]
            image, target = load_train_sample(task)
            phrase = task["phrase"]
            if len(all_train_phrases) > 1 and float(torch.rand(1)) < NEGATIVE_PROB:
                while phrase == task["phrase"]:
                    phrase = all_train_phrases[int(torch.randint(0, len(all_train_phrases), (1,)))]
                target = torch.zeros_like(target)
            support, support_valid = torch.zeros_like(image), False
            if TRAIN_MODE == "phrasecut_plus":
                candidates = [i for i in tasks_by_phrase[phrase] if train_tasks[i]["image_id"] != task["image_id"]]
                if candidates:
                    other = train_tasks[candidates[int(torch.randint(0, len(candidates), (1,)))]]
                    support, support_valid = visual_prompt(*load_train_sample(other)), True
            if PROMPT_AUGMENT:
                prompt = SHUFFLE_PLUS_TEMPLATES[int(torch.randint(0, len(SHUFFLE_PLUS_TEMPLATES), (1,)))].format(phrase)
            else:
                prompt = PROMPT_TEMPLATE.format(phrase=phrase)
            return image, prompt, support, support_valid, target

    def collate_train(samples):
        images, prompts, supports, valid, masks = zip(*samples)
        inputs = dict(processor.tokenizer(text=list(prompts), padding=True, truncation=True, return_tensors="pt"))
        inputs.update(pixel_values=torch.stack(images), support_pixel_values=torch.stack(supports),
                      support_valid=torch.tensor(valid), labels=torch.stack(masks))
        return inputs

    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(PhraseCutPlusDataset(), batch_size=BATCH_SIZE, shuffle=True,
        drop_last=False, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
        collate_fn=collate_train, generator=generator)
    steps_per_epoch = len(train_loader)
    total_steps = EPOCHS * steps_per_epoch
    assert steps_per_epoch == math.ceil(len(train_tasks) / BATCH_SIZE)
    training_metadata = {"enabled": True, "split": "train", "initialization": TRAIN_INIT,
        "seed": SEED, "precision": "fp16_autocast" if USE_AMP else "fp32", "prompt_template": PROMPT_TEMPLATE,
        "base_model_source": model_source, "base_model_revision": MODEL_REVISION if not MODEL_PATH else None,
        "full_valid_train_queries": full_train_queries, "selected_train_queries": len(train_tasks),
        "train_images": len(train_image_ids), "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "steps_per_epoch": steps_per_epoch, "total_planned_steps": total_steps,
        "drop_last": False, "lr": LEARNING_RATE, "min_lr": MIN_LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "num_workers": NUM_WORKERS,
        "train_annotation_sha256": sha256(train_annotation_path),
        "train_manifest_sha256": sha256(OUT / "train_tasks.json"),
        "excluded_train_queries": len(excluded_train), "loss": "BCEWithLogitsLoss",
        "train_mode": TRAIN_MODE, "negative_prob": NEGATIVE_PROB,
        "mix_text_weight": [0.0, MIX_TEXT_MAX] if TRAIN_MODE == "phrasecut_plus" else None,
        "prompt_augment": "shuffle+" if PROMPT_AUGMENT else None, "aug_crop": AUG_CROP,
        "visual_prompt": "crop_blur_highlight352" if TRAIN_MODE == "phrasecut_plus" else None,
        "unique_train_phrases": len(all_train_phrases), "queries_with_support": queries_with_support,
        "protocol": f"{TRAIN_MODE}; frozen CLIP; final epoch checkpoint; eval split held out"}
    write_json(OUT / "training_config.json", training_metadata)
    # Hình kiểm tra 4 mẫu PhraseCut+; fork_rng để không làm đổi RNG của lượt train.
    with torch.random.fork_rng(devices=[]):  # mẫu tạo trên CPU
        torch.manual_seed(SEED)
        preview = [PhraseCutPlusDataset()[i] for i in torch.randperm(len(train_tasks))[:4].tolist()]
    fig, axes = plt.subplots(len(preview), 3, figsize=(9, 3 * len(preview)), squeeze=False)
    for row, (image, prompt, support, valid, target) in zip(axes, preview):
        for ax, value, title in zip(row, [image, support, target], [f"Query: {prompt}", "Support" if valid else "Không có support", "Mask đáp án"]):
            shown = (value * image_std + image_mean).clamp(0, 1).permute(1, 2, 0) if value.ndim == 3 else value
            ax.imshow(shown, cmap="gray" if value.ndim == 2 else None, vmin=0, vmax=1)
            ax.set_title(title, fontsize=9)
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "phrasecut_plus_samples.png", dpi=100)
    plt.close(fig)
    print(f"Mode={TRAIN_MODE}; N_train={len(train_tasks)}; phrases={len(all_train_phrases)}; "
          f"queries with support from another image={queries_with_support}")
    print(f"N_train={len(train_tasks)}, epochs={EPOCHS}, batch={BATCH_SIZE}")
    print(f"steps/epoch={steps_per_epoch}, total optimizer steps={total_steps}")
else:
    print("TRAIN_DECODER=False: đánh giá checkpoint pretrained, không train.")


## 7. Train decoder theo epoch và lưu checkpoint

AdamW + cosine LR từ `LEARNING_RATE` tới `MIN_LEARNING_RATE` trong `total_steps`.
Lưu loss theo epoch và checkpoint sau mỗi epoch, gồm decoder, optimizer, scheduler,
AMP scaler và RNG để resume tại ranh giới epoch. `RESUME_CHECKPOINT` phải có cùng
manifest và cấu hình (kể cả tổng EPOCHS); không tự đổi lịch LR khi tiếp tục.

Checkpoint chỉ lưu decoder vì CLIP đóng băng; ghép với base model/revision cố định để chạy lại.
Không dùng split đánh giá để chọn epoch tốt nhất. Kết quả cuối notebook là model ở **epoch cuối**.


In [ ]:
if TRAIN_DECODER:
    if TRAIN_INIT == "e1_checkpoint":
        if INIT_CHECKPOINT is None:
            found = sorted(p for name in ["decoder_epoch_007.pt", "resume_latest.pt"]
                           for p in Path("/kaggle/input").rglob(name)) if Path("/kaggle/input").exists() else []
            found = [p for p in found if INIT_CHECKPOINT_SHA256 is None or sha256(p) == INIT_CHECKPOINT_SHA256]
            if not found:
                raise FileNotFoundError("Không tìm thấy checkpoint E1 epoch 7 (khớp SHA256) trong Kaggle Input.")
            INIT_CHECKPOINT = str(found[0])  # cùng SHA256 nên bản nào cũng như nhau
        init_sha256 = sha256(Path(INIT_CHECKPOINT))
        if INIT_CHECKPOINT_SHA256 and init_sha256 != INIT_CHECKPOINT_SHA256:
            raise ValueError("Checkpoint E1 khác SHA256 đã chốt.")
        init_state = torch.load(INIT_CHECKPOINT, map_location="cpu", weights_only=True)
        if init_state.get("base_model_id") != MODEL_ID or init_state.get("base_model_revision") != MODEL_REVISION:
            raise ValueError("Checkpoint E1 được train trên base model/revision khác.")
        model.decoder.load_state_dict(init_state["decoder"])
        if ARM == "E4c":
            with torch.no_grad(), torch.random.fork_rng(devices=[device] if device.type == "cuda" else []):
                probe = processor.tokenizer(["the man on the left", "red car"], padding=True, return_tensors="pt").to(device)
                probe_pixels = torch.randn(2, 3, PAPER_SIZE, PAPER_SIZE, device=device)
                before = segment(model, probe_pixels, probe["input_ids"], probe["attention_mask"])
            # Khởi tạo LoRA không được làm đổi RNG lấy mẫu dữ liệu so với B1.
            with torch.random.fork_rng(devices=list(range(torch.cuda.device_count()))):
                attach_lora(model, LORA_RANK, LORA_ALPHA)
            with torch.no_grad():
                after = segment(model, probe_pixels, probe["input_ids"], probe["attention_mask"])
            assert torch.allclose(before, after, atol=1e-4), "LoRA B = 0 nhưng E4c khác E1 lúc khởi tạo"
            print("E4c lúc khởi tạo cho đúng logits của E1 (LoRA B = 0)")
        elif ARM != "B1":
            raise ValueError(f"ARM không hợp lệ: {ARM}")
        new_parameters = sum(p.numel() for p in lora_parameters(model))
        training_metadata.update(arm=ARM, init_checkpoint_sha256=init_sha256, init_epoch=init_state["epoch"],
            init_global_step=init_state["global_step"], decoder_arch="clipseg",
            lora={"rank": LORA_RANK, "alpha": LORA_ALPHA, "targets": ["q_proj", "v_proj"],
                  "layers": f"CLIP vision 0-{max(model.extract_layers)}", "init": "B = 0",
                  "conditioning": "text and support embeddings use frozen CLIP (LoRA disabled)",
                  "grad_checkpoint": LORA_GRAD_CHECKPOINT, "new_parameters": new_parameters,
                  # chỉ ghi khi khác 1.0 để chữ ký cấu hình của các lượt cũ không đổi
                  **({"lr_scale": LORA_LR_SCALE} if LORA_LR_SCALE != 1.0 else {})} if ARM == "E4c" else None)
        write_json(OUT / "training_config.json", training_metadata)
        print(f"ARM={ARM}: từ E1 epoch {init_state['epoch']} (step {init_state['global_step']}), "
              f"+{EPOCHS} epoch, tham số mới {new_parameters:,}")
        del init_state
    elif TRAIN_INIT == "random_decoder":
        model.decoder = type(model.decoder)(model.config)
        model.decoder.post_init()
        model.decoder = model.decoder.to(device)
    if FROM_SCRATCH:
        # Decoder vừa khởi tạo ngẫu nhiên như E1; gắn LoRA (B = 0) mà không đổi RNG lấy mẫu dữ liệu.
        with torch.random.fork_rng(devices=list(range(torch.cuda.device_count()))):
            attach_lora(model, LORA_RANK, LORA_ALPHA)
        new_parameters = sum(p.numel() for p in lora_parameters(model))
        training_metadata.update(arm=ARM, from_scratch=True, decoder_arch="clipseg", schedule=SCRATCH_STAGES,
            lora={"rank": LORA_RANK, "alpha": LORA_ALPHA, "targets": ["q_proj", "v_proj"],
                  "layers": f"CLIP vision 0-{max(model.extract_layers)}", "init": "B = 0",
                  "conditioning": "text and support embeddings use frozen CLIP (LoRA disabled)",
                  "grad_checkpoint": LORA_GRAD_CHECKPOINT, "new_parameters": new_parameters,
                  **({"lr_scale": LORA_LR_SCALE} if LORA_LR_SCALE != 1.0 else {})})
        write_json(OUT / "training_config.json", training_metadata)
        print(f"ARM={ARM} từ đầu: decoder ngẫu nhiên + LoRA ({new_parameters:,} tham số), {EPOCHS} epoch, "
              f"lịch {[(s['epochs'], s['lr'], s['min_lr']) for s in SCRATCH_STAGES]}")
    model.clip.requires_grad_(False)
    model.decoder.requires_grad_(True)
    for parameter in lora_parameters(model):  # E4c: chỉ LoRA trong CLIP được học, trọng số CLIP gốc giữ đóng băng
        parameter.requires_grad_(True)
    optimizer = torch.optim.AdamW(trainable_groups(model, LEARNING_RATE), weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SCRATCH_STAGES[0]["epochs"] * steps_per_epoch if FROM_SCRATCH else total_steps, eta_min=MIN_LEARNING_RATE)
    sync_lora_lr(optimizer)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    signature = hashlib.sha256(json.dumps(training_metadata, sort_keys=True).encode()).hexdigest()
    start_epoch, global_step, history = 0, 0, []
    checkpoint_dir = OUT / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)
    if RESUME_CHECKPOINT == "auto":
        # Checkpoint mới nhất của các phiên trước có cùng chữ ký cấu hình (cùng ARM, lịch, dữ liệu, seed).
        found = []
        for path in (Path("/kaggle/input").rglob("decoder_epoch_*.pt") if Path("/kaggle/input").exists() else []):
            state = torch.load(path, map_location="cpu", weights_only=True)
            if state.get("config_signature") == signature:
                found.append((int(state["epoch"]), str(path)))
            del state
        RESUME_CHECKPOINT = max(found)[1] if found else None
        print("Tiếp tục từ:", RESUME_CHECKPOINT or "không có — bắt đầu epoch 1")
    if not RESUME_CHECKPOINT:
        for old_checkpoint in checkpoint_dir.glob("decoder_epoch_*.pt"):
            old_checkpoint.unlink()
    if RESUME_CHECKPOINT:
        checkpoint = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=True)
        if checkpoint["config_signature"] != signature:
            raise ValueError("Checkpoint khác train manifest/config; giữ nguyên cấu hình khi resume.")
        model.decoder.load_state_dict(checkpoint["decoder"])
        if checkpoint.get("lora"):
            load_lora_state(model, checkpoint["lora"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        if checkpoint.get("numerical_recovery"):
            save_numerical_report(OUT, checkpoint["numerical_recovery"])
        start_epoch, global_step, history = checkpoint["epoch"], checkpoint["global_step"], checkpoint["history"]
        generator.set_state(checkpoint["loader_rng"].cpu())
        torch.set_rng_state(checkpoint["torch_rng"].cpu())
        random.setstate(checkpoint["python_rng"])
        state = checkpoint["numpy_rng"]
        np.random.set_state((state[0], state[1].numpy().astype(np.uint32), state[2], state[3], state[4]))
        if device.type == "cuda" and checkpoint["cuda_rng"] is not None:
            # Session resume có thể khác số GPU; chỉ khôi phục các GPU đang có.
            for index, value in enumerate(checkpoint["cuda_rng"][:torch.cuda.device_count()]):
                torch.cuda.set_rng_state(value.cpu(), index)

    class TrainForward(torch.nn.Module):
        """Một bước forward train. DataParallel chia batch của module này cho từng GPU."""
        def __init__(self, model):
            super().__init__()
            self.model = model

        def forward(self, pixel_values, input_ids, attention_mask, support_pixel_values=None, text_weights=None, force_fp32=False):
            size = pixel_values.shape[0]
            context = torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP and not force_fp32)
            with context:
                # segment giữ [B,H,W] để DataParallel gộp đúng; E4c tính activation CLIP có LoRA với gradient.
                return segment(self.model, pixel_values, input_ids, attention_mask, support_pixel_values, text_weights)

    train_gpus = torch.cuda.device_count() if device.type == "cuda" else 0
    use_data_parallel = MULTI_GPU and train_gpus > 1
    train_forward = TrainForward(model)
    if use_data_parallel:
        train_forward = torch.nn.DataParallel(train_forward)
        assert BATCH_SIZE >= train_gpus, "BATCH_SIZE phải ≥ số GPU."
    print(f"Train trên {train_gpus if use_data_parallel else 1} {'GPU' if device.type == 'cuda' else 'CPU'}"
          f"{' (DataParallel, mỗi GPU ~' + str(math.ceil(BATCH_SIZE / train_gpus)) + ' mẫu/batch)' if use_data_parallel else ''}")
    last_epoch = EPOCHS if STOP_AFTER_EPOCH is None else min(EPOCHS, STOP_AFTER_EPOCH)
    STOPPED_EARLY = last_epoch < EPOCHS
    boundaries = [sum(s["epochs"] for s in SCRATCH_STAGES[:i]) for i in range(1, len(SCRATCH_STAGES))] if FROM_SCRATCH else []
    for epoch in range(start_epoch, last_epoch):
        if epoch in boundaries:
            stage = SCRATCH_STAGES[boundaries.index(epoch) + 1]
            stage_steps = stage["epochs"] * steps_per_epoch
            if stage["start"].startswith("new optimizer"):
                # Như lúc bắt đầu B1: optimizer, AMP scaler mới và thứ tự dữ liệu từ SEED.
                optimizer = torch.optim.AdamW(trainable_groups(model, stage["lr"]), weight_decay=WEIGHT_DECAY)
                scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
                generator.manual_seed(SEED)
            else:
                # Như lượt train tiếp E1 epoch 6-7: giữ trạng thái AdamW, đặt lại lr và lịch cosine.
                for index, group in enumerate(optimizer.param_groups):
                    group["lr"] = group["initial_lr"] = stage["lr"] * (LORA_LR_SCALE if index else 1.0)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=stage_steps, eta_min=stage["min_lr"])
            sync_lora_lr(optimizer)
            print(f"Epoch {epoch + 1}: chặng mới {stage}", flush=True)
        model.train()
        model.clip.eval()
        model.decoder.train()
        loss_sum, seen = 0.0, 0
        for index in range(train_gpus):
            torch.cuda.reset_peak_memory_stats(index)
        epoch_started = time.perf_counter()
        for batch in tqdm(train_loader, desc=f"Train epoch {epoch + 1}/{EPOCHS}"):
            optimizer.zero_grad(set_to_none=True)
            inputs = {key: batch[key] for key in ["pixel_values", "input_ids", "attention_mask"]}
            if TRAIN_MODE == "phrasecut_plus":
                inputs["support_pixel_values"] = batch["support_pixel_values"]
                inputs["text_weights"] = sample_text_weights(batch["support_valid"], MIX_TEXT_MAX)
            if not use_data_parallel:
                # DataParallel tự chép từng phần batch từ CPU sang đúng GPU; một GPU thì chép trước.
                inputs = {key: value.to(device, non_blocking=True) for key, value in inputs.items()}
            old_scale = scaler.get_scale()
            loss = loss_backward_with_recovery(train_forward, model, inputs, batch["labels"], scaler,
                OUT, epoch + 1, seen // BATCH_SIZE, global_step)
            scaler.step(optimizer)
            scaler.update()
            # AMP có thể bỏ optimizer step khi overflow; không tăng lịch LR lúc đó.
            if not USE_AMP or scaler.get_scale() >= old_scale:
                scheduler.step()
                sync_lora_lr(optimizer)
                global_step += 1
            count = batch["labels"].shape[0]
            loss_sum += loss.detach().item() * count
            seen += count
        assert seen == len(train_tasks), "Epoch chưa duyệt đủ train queries."
        synchronize()
        history.append({"epoch": epoch + 1, "train_loss": loss_sum / seen, "n_queries": seen,
            "optimizer_steps_total": global_step, "learning_rate_next": optimizer.param_groups[0]["lr"],
            "epoch_seconds": time.perf_counter() - epoch_started, "gpus": train_gpus if use_data_parallel else min(train_gpus, 1),
            "peak_gpu_mb": [round(torch.cuda.max_memory_allocated(i) / 1024**2) for i in range(train_gpus)]})
        np_state = np.random.get_state()
        checkpoint = {"epoch": epoch + 1, "global_step": global_step, "config_signature": signature,
            "numerical_policy": NUMERICAL_POLICY, "numerical_recovery": json.loads((OUT / "numerical_recovery.json").read_text()) if (OUT / "numerical_recovery.json").exists() else None, "decoder": model.decoder.state_dict(), "lora": lora_state_dict(model), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "history": history,
            "loader_rng": generator.get_state(), "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
            "python_rng": random.getstate(),
            "numpy_rng": (np_state[0], torch.from_numpy(np_state[1].astype(np.int64)), np_state[2], np_state[3], np_state[4]),
            "base_model_id": MODEL_ID, "base_model_revision": MODEL_REVISION,
            "training": training_metadata}  # E2/E3 đọc epoch, batch size từ đây
        checkpoint_path = checkpoint_dir / f"decoder_epoch_{epoch + 1:03d}.pt"
        torch.save(checkpoint, checkpoint_path)
        write_csv(OUT / "training_history.csv", history)
        print(history[-1])
    assert start_epoch <= EPOCHS, "Checkpoint vượt EPOCHS được yêu cầu."
    model.eval()
    training_metadata.update({"completed_epochs": last_epoch, "optimizer_steps_completed": global_step,
        "final_checkpoint": str(RESUME_CHECKPOINT) if start_epoch >= last_epoch else str(checkpoint_path)})
    write_json(OUT / "training_config.json", training_metadata)
    # Backbone không nhận gradient; chỉ decoder có tham số trainable.
    assert all(p.grad is None for n, p in model.clip.named_parameters() if "lora_" not in n)
    print("LR cuối:", [round(g["lr"], 10) for g in optimizer.param_groups], flush=True)
    if STOPPED_EARLY:
        print(f"Dừng sau epoch {last_epoch}/{EPOCHS}. Thêm Output phiên này làm Input rồi chạy phiên tiếp; "
              "chưa chấm val/test.")
    print(f"Training complete. Evaluate final epoch on {EVAL_SPLIT} next.")
elif DECODER_CHECKPOINT:
    # Chỉ đánh giá: thay decoder tác giả bằng decoder đã train trong một lượt E1 trước.
    checkpoint = torch.load(DECODER_CHECKPOINT, map_location="cpu", weights_only=True)
    if checkpoint.get("base_model_id") != MODEL_ID or checkpoint.get("base_model_revision") != MODEL_REVISION:
        raise ValueError("Checkpoint được train trên base model/revision khác MODEL_ID/MODEL_REVISION.")
    if checkpoint.get("lora"):
        adapter = (checkpoint.get("training") or {}).get("lora") or {}
        attach_lora(model, adapter.get("rank", LORA_RANK), adapter.get("alpha", LORA_ALPHA))
        load_lora_state(model, checkpoint["lora"])
    model.decoder.load_state_dict(checkpoint["decoder"])
    model.eval()
    training_metadata = {"enabled": False, "decoder_checkpoint": str(DECODER_CHECKPOINT),
        "decoder_checkpoint_sha256": sha256(Path(DECODER_CHECKPOINT)), "decoder_epoch": checkpoint["epoch"],
        "decoder_training": checkpoint.get("training"), "decoder_history": checkpoint.get("history")}
    del checkpoint
    print(f"Đánh giá decoder epoch {training_metadata['decoder_epoch']} từ {DECODER_CHECKPOINT}")
else:
    print("Đánh giá decoder pretrained của tác giả (CIDAS/clipseg-rd64-refined).")


## 8. Inference toàn bộ query đã chọn của split đánh giá

Warm-up trước khi đo. Batch các phrase trên cùng ảnh. Mỗi query được chấm hai cách: theo paper
(352×352, 49 ngưỡng, lưu `paper_threshold_stats.npz` để tính lại được) và ở độ phân giải gốc.
Lưu mask theo `task_id.png`, đúng quy ước PhraseCut; GT, ảnh phân tích và probability tùy chọn
được ghi riêng. CSV được checkpoint sau mỗi ảnh; nếu lỗi, kết quả một phần giữ lại nhưng
chưa có `summary.json` hoàn chỉnh. Chạy lại cell sẽ tính lại toàn bộ để tránh ghép nhầm lượt chạy.


In [ ]:
if not STOPPED_EARLY:
    for name in ["masks", "ground_truth", "visualizations", "probabilities"]:
        (OUT / name).mkdir(exist_ok=True)
    # Tách từng run bằng config/manifest: không trộn artifact cũ vào ZIP.
    for folder in ["masks", "ground_truth", "visualizations", "probabilities"]:
        for path in (OUT / folder).iterdir():
            if path.is_file():
                path.unlink()
    for filename in ["summary.json", "sample_metrics.csv", "calibration_thresholds.csv", "run.json"]:
        (OUT / filename).unlink(missing_ok=True)
    
    tasks_by_image = defaultdict(list)
    for task in selected:
        tasks_by_image[int(task["image_id"])].append(task)
    first_image_id = image_ids[0]
    with Image.open(image_paths[first_image_id]) as raw:
        warmup_image = raw.convert("RGB")
    warmup_tasks = tasks_by_image[first_image_id][:BATCH_SIZE]
    for _ in range(2):
        predict_batch(warmup_image, warmup_tasks)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    
    def save_visualization(image, target, probability, task, metrics, index):
        prediction = probability >= THRESHOLD
        overlay = np.asarray(image).astype(np.float32).copy()
        overlay[prediction] = 0.55 * overlay[prediction] + 0.45 * np.array([255, 60, 60])
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(image); axes[0].set_title("Ảnh")
        axes[1].imshow(target, cmap="gray", vmin=0, vmax=1); axes[1].set_title("Ground truth")
        axes[2].imshow(probability, cmap="magma", vmin=0, vmax=1); axes[2].set_title("Sigmoid probability")
        axes[3].imshow(overlay.astype(np.uint8)); axes[3].set_title(f"t={THRESHOLD}, IoU={metrics['iou']:.3f}")
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"{task['task_id']} — {task['phrase']}")
        fig.tight_layout()
        fig.savefig(OUT / "visualizations" / f"{index:02d}_{task['task_id']}.png", dpi=120, bbox_inches="tight")
        plt.close(fig)
    
    rows, sweep_rows, batch_times, paper_stats = [], [], [], []
    visual_count = 0
    started_all = time.perf_counter()
    for image_id in tqdm(image_ids, desc="CLIPSeg images"):
        with Image.open(image_paths[image_id]) as raw:
            image = raw.convert("RGB")
        current_tasks = tasks_by_image[image_id]
        for start in range(0, len(current_tasks), BATCH_SIZE):
            batch = current_tasks[start:start + BATCH_SIZE]
            probabilities, probabilities_352, seconds = predict_batch(image, batch, with_paper=True)
            assert probabilities_352.shape[-2:] == (PAPER_SIZE, PAPER_SIZE), probabilities_352.shape
            batch_times.append({"queries": len(batch), "forward_resize_seconds": seconds})
            for task, probability, probability_352 in zip(batch, probabilities, probabilities_352):
                target = rasterize_gt(task, image.width, image.height)
                paper_stats.append(paper_threshold_stats(probability_352, paper_target(target)))
                base = {"task_id": str(task["task_id"]), "image_id": image_id,
                    "phrase": task["phrase"], "prompt": PROMPT_TEMPLATE.format(phrase=task["phrase"]),
                    "group": group_for_image[image_id], "width": image.width, "height": image.height}
                metrics = binary_metrics(probability, target, THRESHOLD)
                rows.append({**base, "threshold": THRESHOLD, **metrics,
                    "pixel_ap": float(average_precision_score(target.ravel(), probability.ravel()))})
                for threshold in THRESHOLD_GRID:
                    sweep_rows.append({**base, "threshold": threshold, **binary_metrics(probability, target, threshold)})
                Image.fromarray((probability >= THRESHOLD).astype(np.uint8) * 255).save(OUT / "masks" / f"{task['task_id']}.png")
                Image.fromarray(target.astype(np.uint8) * 255).save(OUT / "ground_truth" / f"{task['task_id']}.png")
                if SAVE_PROBABILITIES:
                    np.save(OUT / "probabilities" / f"{task['task_id']}.npy", probability)
                if visual_count < N_VISUALIZATIONS:
                    save_visualization(image, target, probability, task, metrics, visual_count)
                    visual_count += 1
        write_csv(OUT / "sample_metrics.csv", rows)
    elapsed_all = time.perf_counter() - started_all
    assert len(rows) == len(selected) == len(paper_stats), "Chưa đánh giá đủ query."
    paper_stats = np.stack(paper_stats)
    np.savez_compressed(OUT / "paper_threshold_stats.npz", stats=paper_stats, thresholds=PAPER_THRESHOLDS,
                        task_ids=np.array([r["task_id"] for r in rows]))
    print(f"Completed {len(rows)} queries; wall time {elapsed_all:.1f}s")


## 9. Tổng hợp và chọn ngưỡng trên calibration

`paper_protocol` là số so được với bảng paper (`miou_0.3`, `fgiou_0.3`, `ap`; paper báo CLIPSeg PC+ trên test:
43,4 / 54,7 / 76,7). `selected_fixed_threshold` là metric ở độ phân giải gốc, t=0,3, cho toàn bộ query đã chọn.
Ngưỡng bổ sung được chọn bằng mean IoU **chỉ trên calibration**, rồi báo riêng trên evaluation.
Không chọn template hoặc threshold theo evaluation. Nếu `MAX_QUERIES` khác None,
ghi đây là tập con, không gắn nhãn kết quả toàn split.


In [ ]:
if not STOPPED_EARLY:
    calibration_table = []
    for threshold in THRESHOLD_GRID:
        subset = [r for r in sweep_rows if r["group"] == "calibration" and r["threshold"] == threshold]
        calibration_table.append({"threshold": threshold, **summarize(subset)})
    best_threshold = max(calibration_table, key=lambda r: (r["mean_sample_iou"], -r["threshold"]))["threshold"]
    write_csv(OUT / "calibration_thresholds.csv", calibration_table)
    evaluation_fixed = [r for r in rows if r["group"] == "evaluation"]
    evaluation_tuned = [r for r in sweep_rows if r["group"] == "evaluation" and r["threshold"] == best_threshold]
    summary = {
        "training": training_metadata,
        "dataset": "VGPhraseCut", "split": EVAL_SPLIT, "complete_official_split": len(selected) == len(tasks),
        "threshold": THRESHOLD, "calibration_selected_threshold": best_threshold,
        "selected_fixed_threshold": summarize(rows),
        "calibration_fixed_threshold": summarize([r for r in rows if r["group"] == "calibration"]),
        "evaluation_fixed_threshold": summarize(evaluation_fixed),
        "evaluation_calibrated_threshold": summarize(evaluation_tuned),
        "paper_protocol": paper_metrics(paper_stats),
        "paper_reference_test": {"clipseg_pc_plus": {"miou_0.3": 0.434, "fgiou_0.3": 0.547, "ap": 0.767},
                                 "clipseg_pc": {"miou_0.3": 0.461, "fgiou_0.3": 0.562, "ap": 0.782}},
        "metric_note": ("paper_protocol follows CLIPSeg score.py/FixedIntervalMetrics at 352x352 (49 thresholds, p > t); "
                        "other blocks are at original resolution: IoU averaged per query, pixel AP averaged per query."),
    }
    write_json(OUT / "summary.json", summary)
    inference_seconds = sum(r["forward_resize_seconds"] for r in batch_times)
    run = {
        "training": training_metadata,
        "model_id": MODEL_ID, "requested_revision": MODEL_REVISION if not MODEL_PATH else None,
        "model_source": model_source, "resolved_commit": getattr(model.config, "_commit_hash", None),
        "phrasecut_api_commit": PHRASECUT_API_COMMIT,
        "annotation_sha256": sha256(annotation_path), "image_metadata_sha256": sha256(info_path),
        "selected_tasks_sha256": sha256(OUT / "selected_tasks.json"),
        "seed": SEED, "max_queries": MAX_QUERIES, "batch_size": BATCH_SIZE,
        "prompt_template": PROMPT_TEMPLATE, "threshold": THRESHOLD, "threshold_grid": THRESHOLD_GRID,
        "calibration_fraction": CALIBRATION_FRACTION, "calibration_image_ids": sorted(calibration_ids),
        "evaluation_image_ids": sorted(set(image_ids) - calibration_ids),
        "precision": "fp16_autocast" if EVAL_AMP else "fp32", "train_precision": "fp16_autocast" if USE_AMP else "fp32",
        "device": str(device),
        "gpu_name": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "torch_cuda_version": torch.version.cuda, "python": platform.python_version(), "versions": versions,
        "image_processor": processor.image_processor.to_dict(), "model_config": model.config.to_dict(),
        "parameter_count": sum(p.numel() for p in model.parameters()),
        "decoder_parameter_count": sum(p.numel() for p in model.decoder.parameters()),
        "forward_resize_seconds_total": inference_seconds,
        "forward_resize_ms_per_query": 1000 * inference_seconds / len(rows),
        "wall_seconds_inference_metrics_saving": elapsed_all,
        "peak_allocated_gpu_mb": torch.cuda.max_memory_allocated(device) / 1024**2 if device.type == "cuda" else None,
        "timing_note": "Two warmup batches; forward+resize+sigmoid timed with CUDA sync; excludes preprocessing, transfer and metric/save. GPU peak includes resizing.",
        "save_probabilities": SAVE_PROBABILITIES,
    }
    write_json(OUT / "run.json", run)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    paper = summary["paper_protocol"]
    print(f"Theo protocol paper ({EVAL_SPLIT}): mIoU {100 * paper['miou_0.3']:.1f} | IoU_FG {100 * paper['fgiou_0.3']:.1f} | "
          f"AP {100 * paper['ap']:.1f}  (paper, CLIPSeg PC+ trên test: 43.4 | 54.7 | 76.7)")
    print(f"Forward+resize+sigmoid: {run['forward_resize_ms_per_query']:.2f} ms/query")


## 9b. Lưu kết quả theo split, rồi chấm test

Kết quả của split vừa chấm được chép vào `splits/<split>/` (summary, metric từng query, stats theo protocol paper)
để `scripts/compare_e4.py` so ghép cặp với B1. Sau val, notebook chấm tiếp **test** bằng cùng checkpoint epoch cuối:
test chỉ để báo cáo, không dùng để chọn gì.


In [ ]:
if not STOPPED_EARLY:
    split_dir = OUT / "splits" / EVAL_SPLIT
    split_dir.mkdir(parents=True, exist_ok=True)
    for name in ["summary.json", "run.json", "sample_metrics.csv", "calibration_thresholds.csv",
                 "paper_threshold_stats.npz", "selected_tasks.json", "image_errors.json"]:
        if (OUT / name).is_file():
            shutil.copy2(OUT / name, split_dir / name)
    if MAX_QUERIES is None:
        assert summary["complete_official_split"], "Chưa chấm đủ split chính thức."
    paper = summary["paper_protocol"]
    print(f"Đã lưu {split_dir}: {paper['n_queries']} query, mIoU {100 * paper['miou_0.3']:.2f} | "
          f"IoU_FG {100 * paper['fgiou_0.3']:.2f} | AP {100 * paper['ap']:.2f}")


In [ ]:
if not STOPPED_EARLY:
    elapsed_hours = (_time.time() - SESSION_STARTED) / 3600
    RUN_TEST = TEST_TIME_BUDGET_HOURS is None or elapsed_hours <= TEST_TIME_BUDGET_HOURS
    write_json(OUT / "session_time.json", {"elapsed_hours_after_val": elapsed_hours,
                                             "test_time_budget_hours": TEST_TIME_BUDGET_HOURS, "run_test": RUN_TEST})
    if RUN_TEST:
        EVAL_SPLIT = "test"
        print(f"Chấm test bằng checkpoint epoch cuối (đã chạy {elapsed_hours:.2f} giờ)")
    else:
        print(f"BỎ CHẤM TEST: đã chạy {elapsed_hours:.2f} giờ > {TEST_TIME_BUDGET_HOURS} giờ. "
              "Kết quả val và checkpoint vẫn được lưu; chấm test sau bằng chế độ chỉ đánh giá.")


In [ ]:
if not STOPPED_EARLY and RUN_TEST:
    # Drive ID của annotation chính thức, theo download_dataset.py của PhraseCut.
    REFER_DRIVE_IDS = {"miniv": "1oLcTQ1blTIQyu5ZMelQN2HSniuQaxM4E", "val": "1UyojArOFPlsSeNbA9fHWjCjOOU-OCohG",
                       "test": "1jrzXm1gcq6f5hNDeamZd0UmyyHUv61IZ", "train": "1qx-0q6r9r0YUGpoyT0B8HJKmUFWQDSu7"}
    
    def metadata_path(root):
        for name in ["image_data_split.json", "image_data_split3000.json"]:
            candidate = root / name
            if candidate.is_file():
                return candidate
        return root / "image_data_split.json"
    
    if DATA_ROOT is None:
        candidates = sorted(Path("/kaggle/input").rglob(f"refer_{EVAL_SPLIT}.json")) if Path("/kaggle/input").exists() else []
        candidates = [p.parent for p in candidates if metadata_path(p.parent).is_file()]
        if len(candidates) > 1:
            raise ValueError(f"Có nhiều dataset chứa refer_{EVAL_SPLIT}.json; đặt DATA_ROOT rõ ràng: " + str(candidates))
        data_root = candidates[0] if candidates else WORK_ROOT / "phrasecut_annotations"
    else:
        data_root = Path(DATA_ROOT)
        if not data_root.is_dir():
            raise FileNotFoundError(f"DATA_ROOT không tồn tại: {data_root}")
    data_root.mkdir(parents=True, exist_ok=True)
    annotation_path = data_root / f"refer_{EVAL_SPLIT}.json"
    info_path = metadata_path(data_root)
    
    def ensure_json(path, file_id):
        if path.is_file():
            value = json.loads(path.read_text(encoding="utf-8"))
            if not isinstance(value, list) or not value:
                raise ValueError(f"JSON không phải danh sách annotation: {path}")
            return value
        if str(path).startswith("/kaggle/input/"):
            raise FileNotFoundError(f"Thiếu {path.name} trong input read-only; bổ sung Kaggle Dataset.")
        temp = path.with_suffix(".download")
        try:
            result = gdown.download(id=file_id, output=str(temp), quiet=False)
            if result is None:
                raise RuntimeError("Google Drive không trả về file.")
            value = json.loads(temp.read_text(encoding="utf-8"))
            if not isinstance(value, list) or not value:
                raise ValueError("Nội dung download không phải annotation JSON.")
            temp.replace(path)
            return value
        except Exception as exc:
            raise RuntimeError(f"Không tải được {path.name}. Bật Internet, hoặc Add Input dataset và đặt DATA_ROOT.") from exc
    
    tasks = ensure_json(annotation_path, REFER_DRIVE_IDS[EVAL_SPLIT])
    image_info = ensure_json(info_path, "1xB9eExJo35K3DQu8PnBWZ1q_OMoMl8-i")
    info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == EVAL_SPLIT}
    assert len({str(t["task_id"]) for t in tasks}) == len(tasks), "Task ID bị trùng."
    assert all(int(t["image_id"]) in info_by_id for t in tasks), f"Annotation không thuộc split {EVAL_SPLIT}."
    assert all(t.get("Polygons") and t.get("phrase", "").strip() for t in tasks), "Thiếu polygon GT hoặc phrase."
    assert all(re.fullmatch(r"[A-Za-z0-9_-]+", str(t["task_id"])) for t in tasks), "Task ID không hợp lệ."
    
    selected = sorted(tasks, key=lambda t: str(t["task_id"]))
    if MAX_QUERIES is not None and MAX_QUERIES < len(selected):
        selected = random.Random(SEED).sample(selected, MAX_QUERIES)
    selected.sort(key=lambda t: (int(t["image_id"]), str(t["task_id"])))
    image_ids = sorted({int(t["image_id"]) for t in selected})
    if len(image_ids) < 2:
        raise ValueError("Cần ít nhất 2 image_id để chia calibration/evaluation; tăng MAX_QUERIES.")
    shuffled_ids = image_ids.copy()
    random.Random(SEED).shuffle(shuffled_ids)
    n_cal = min(len(image_ids) - 1, max(1, round(len(image_ids) * CALIBRATION_FRACTION)))
    calibration_ids = set(shuffled_ids[:n_cal])
    group_for_image = {i: "calibration" if i in calibration_ids else "evaluation" for i in image_ids}
    assert not calibration_ids.intersection(set(image_ids) - calibration_ids)
    write_json(OUT / "selected_tasks.json", [{"task_id": str(t["task_id"]), "image_id": int(t["image_id"]),
        "phrase": t["phrase"], "group": group_for_image[int(t["image_id"])]} for t in selected])
    print(f"Annotation {EVAL_SPLIT}: {len(tasks)} queries / {len(info_by_id)} images")
    print(f"Selected: {len(selected)} queries / {len(image_ids)} images; calibration {n_cal} images")


In [ ]:
if not STOPPED_EARLY and RUN_TEST:
    image_cache = WORK_ROOT / "phrasecut_image_cache"
    image_cache.mkdir(parents=True, exist_ok=True)
    def image_search_roots(*extra):
        """Nơi tìm ảnh <image_id>.jpg: IMAGE_ROOT nếu đặt, nếu không thì mọi Kaggle Input; cộng thư mục annotation."""
        roots = [Path(IMAGE_ROOT)] if IMAGE_ROOT else ([Path("/kaggle/input")] if Path("/kaggle/input").exists() else [])
        roots += [Path(r) for r in extra]
        # Bỏ thư mục nằm trong thư mục đã quét để không quét hai lần.
        return [r for i, r in enumerate(roots) if r.is_dir() and not any(r == q or q in r.parents for q in roots[:i])]
    
    def index_local_images(roots, wanted_ids):
        index = {}
        for search_root in roots:
            for path in sorted(search_root.rglob("*")):
                if path.suffix.lower() in {".jpg", ".jpeg", ".png"} and path.stem.isdigit() and int(path.stem) in wanted_ids:
                    index.setdefault(int(path.stem), path)
        return index
    
    local_images = index_local_images(image_search_roots(data_root), set(image_ids))
    print(f"Ảnh có sẵn trong Input: {len(local_images)}/{len(image_ids)}; phần còn lại tải từ Visual Genome")
    
    def valid_image(path, info):
        with Image.open(path) as image:
            image.load()
            if image.size != (int(info["width"]), int(info["height"])):
                raise ValueError(f"{path.name}: ảnh {image.size} != metadata {(info['width'], info['height'])}")
    
    def prepare_image(image_id):
        info = info_by_id[image_id]
        path = local_images.get(image_id, image_cache / f"{image_id}.jpg")
        if path.is_file():
            valid_image(path, info)
            return image_id, path
        url = info["url"].replace("http://", "https://", 1)
        temp = path.with_suffix(".download")
        for attempt in range(3):
            try:
                with requests.get(url, stream=True, timeout=(10, 60)) as response:
                    response.raise_for_status()
                    with temp.open("wb") as f:
                        for chunk in response.iter_content(1024 * 1024):
                            f.write(chunk)
                valid_image(temp, info)
                temp.replace(path)
                return image_id, path
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(attempt + 1)
    
    image_paths, image_errors = {}, []
    def prepare_safe(image_id):
        try:
            return prepare_image(image_id), None
        except Exception as exc:
            return None, {"image_id": image_id, "error": str(exc)}
    
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        for result, error in tqdm(pool.map(prepare_safe, image_ids), total=len(image_ids), desc="Images"):
            if error:
                image_errors.append(error)
            else:
                image_paths[result[0]] = result[1]
    write_json(OUT / "image_errors.json", image_errors)
    if image_errors:
        raise RuntimeError(f"{len(image_errors)} ảnh lỗi; xem image_errors.json. Sửa dữ liệu rồi chạy lại.")
    print("Validated images:", len(image_paths))


In [ ]:
if not STOPPED_EARLY and RUN_TEST:
    for name in ["masks", "ground_truth", "visualizations", "probabilities"]:
        (OUT / name).mkdir(exist_ok=True)
    # Tách từng run bằng config/manifest: không trộn artifact cũ vào ZIP.
    for folder in ["masks", "ground_truth", "visualizations", "probabilities"]:
        for path in (OUT / folder).iterdir():
            if path.is_file():
                path.unlink()
    for filename in ["summary.json", "sample_metrics.csv", "calibration_thresholds.csv", "run.json"]:
        (OUT / filename).unlink(missing_ok=True)
    
    tasks_by_image = defaultdict(list)
    for task in selected:
        tasks_by_image[int(task["image_id"])].append(task)
    first_image_id = image_ids[0]
    with Image.open(image_paths[first_image_id]) as raw:
        warmup_image = raw.convert("RGB")
    warmup_tasks = tasks_by_image[first_image_id][:BATCH_SIZE]
    for _ in range(2):
        predict_batch(warmup_image, warmup_tasks)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    
    def save_visualization(image, target, probability, task, metrics, index):
        prediction = probability >= THRESHOLD
        overlay = np.asarray(image).astype(np.float32).copy()
        overlay[prediction] = 0.55 * overlay[prediction] + 0.45 * np.array([255, 60, 60])
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(image); axes[0].set_title("Ảnh")
        axes[1].imshow(target, cmap="gray", vmin=0, vmax=1); axes[1].set_title("Ground truth")
        axes[2].imshow(probability, cmap="magma", vmin=0, vmax=1); axes[2].set_title("Sigmoid probability")
        axes[3].imshow(overlay.astype(np.uint8)); axes[3].set_title(f"t={THRESHOLD}, IoU={metrics['iou']:.3f}")
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"{task['task_id']} — {task['phrase']}")
        fig.tight_layout()
        fig.savefig(OUT / "visualizations" / f"{index:02d}_{task['task_id']}.png", dpi=120, bbox_inches="tight")
        plt.close(fig)
    
    rows, sweep_rows, batch_times, paper_stats = [], [], [], []
    visual_count = 0
    started_all = time.perf_counter()
    for image_id in tqdm(image_ids, desc="CLIPSeg images"):
        with Image.open(image_paths[image_id]) as raw:
            image = raw.convert("RGB")
        current_tasks = tasks_by_image[image_id]
        for start in range(0, len(current_tasks), BATCH_SIZE):
            batch = current_tasks[start:start + BATCH_SIZE]
            probabilities, probabilities_352, seconds = predict_batch(image, batch, with_paper=True)
            assert probabilities_352.shape[-2:] == (PAPER_SIZE, PAPER_SIZE), probabilities_352.shape
            batch_times.append({"queries": len(batch), "forward_resize_seconds": seconds})
            for task, probability, probability_352 in zip(batch, probabilities, probabilities_352):
                target = rasterize_gt(task, image.width, image.height)
                paper_stats.append(paper_threshold_stats(probability_352, paper_target(target)))
                base = {"task_id": str(task["task_id"]), "image_id": image_id,
                    "phrase": task["phrase"], "prompt": PROMPT_TEMPLATE.format(phrase=task["phrase"]),
                    "group": group_for_image[image_id], "width": image.width, "height": image.height}
                metrics = binary_metrics(probability, target, THRESHOLD)
                rows.append({**base, "threshold": THRESHOLD, **metrics,
                    "pixel_ap": float(average_precision_score(target.ravel(), probability.ravel()))})
                for threshold in THRESHOLD_GRID:
                    sweep_rows.append({**base, "threshold": threshold, **binary_metrics(probability, target, threshold)})
                Image.fromarray((probability >= THRESHOLD).astype(np.uint8) * 255).save(OUT / "masks" / f"{task['task_id']}.png")
                Image.fromarray(target.astype(np.uint8) * 255).save(OUT / "ground_truth" / f"{task['task_id']}.png")
                if SAVE_PROBABILITIES:
                    np.save(OUT / "probabilities" / f"{task['task_id']}.npy", probability)
                if visual_count < N_VISUALIZATIONS:
                    save_visualization(image, target, probability, task, metrics, visual_count)
                    visual_count += 1
        write_csv(OUT / "sample_metrics.csv", rows)
    elapsed_all = time.perf_counter() - started_all
    assert len(rows) == len(selected) == len(paper_stats), "Chưa đánh giá đủ query."
    paper_stats = np.stack(paper_stats)
    np.savez_compressed(OUT / "paper_threshold_stats.npz", stats=paper_stats, thresholds=PAPER_THRESHOLDS,
                        task_ids=np.array([r["task_id"] for r in rows]))
    print(f"Completed {len(rows)} queries; wall time {elapsed_all:.1f}s")


In [ ]:
if not STOPPED_EARLY and RUN_TEST:
    calibration_table = []
    for threshold in THRESHOLD_GRID:
        subset = [r for r in sweep_rows if r["group"] == "calibration" and r["threshold"] == threshold]
        calibration_table.append({"threshold": threshold, **summarize(subset)})
    best_threshold = max(calibration_table, key=lambda r: (r["mean_sample_iou"], -r["threshold"]))["threshold"]
    write_csv(OUT / "calibration_thresholds.csv", calibration_table)
    evaluation_fixed = [r for r in rows if r["group"] == "evaluation"]
    evaluation_tuned = [r for r in sweep_rows if r["group"] == "evaluation" and r["threshold"] == best_threshold]
    summary = {
        "training": training_metadata,
        "dataset": "VGPhraseCut", "split": EVAL_SPLIT, "complete_official_split": len(selected) == len(tasks),
        "threshold": THRESHOLD, "calibration_selected_threshold": best_threshold,
        "selected_fixed_threshold": summarize(rows),
        "calibration_fixed_threshold": summarize([r for r in rows if r["group"] == "calibration"]),
        "evaluation_fixed_threshold": summarize(evaluation_fixed),
        "evaluation_calibrated_threshold": summarize(evaluation_tuned),
        "paper_protocol": paper_metrics(paper_stats),
        "paper_reference_test": {"clipseg_pc_plus": {"miou_0.3": 0.434, "fgiou_0.3": 0.547, "ap": 0.767},
                                 "clipseg_pc": {"miou_0.3": 0.461, "fgiou_0.3": 0.562, "ap": 0.782}},
        "metric_note": ("paper_protocol follows CLIPSeg score.py/FixedIntervalMetrics at 352x352 (49 thresholds, p > t); "
                        "other blocks are at original resolution: IoU averaged per query, pixel AP averaged per query."),
    }
    write_json(OUT / "summary.json", summary)
    inference_seconds = sum(r["forward_resize_seconds"] for r in batch_times)
    run = {
        "training": training_metadata,
        "model_id": MODEL_ID, "requested_revision": MODEL_REVISION if not MODEL_PATH else None,
        "model_source": model_source, "resolved_commit": getattr(model.config, "_commit_hash", None),
        "phrasecut_api_commit": PHRASECUT_API_COMMIT,
        "annotation_sha256": sha256(annotation_path), "image_metadata_sha256": sha256(info_path),
        "selected_tasks_sha256": sha256(OUT / "selected_tasks.json"),
        "seed": SEED, "max_queries": MAX_QUERIES, "batch_size": BATCH_SIZE,
        "prompt_template": PROMPT_TEMPLATE, "threshold": THRESHOLD, "threshold_grid": THRESHOLD_GRID,
        "calibration_fraction": CALIBRATION_FRACTION, "calibration_image_ids": sorted(calibration_ids),
        "evaluation_image_ids": sorted(set(image_ids) - calibration_ids),
        "precision": "fp16_autocast" if EVAL_AMP else "fp32", "train_precision": "fp16_autocast" if USE_AMP else "fp32",
        "device": str(device),
        "gpu_name": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "torch_cuda_version": torch.version.cuda, "python": platform.python_version(), "versions": versions,
        "image_processor": processor.image_processor.to_dict(), "model_config": model.config.to_dict(),
        "parameter_count": sum(p.numel() for p in model.parameters()),
        "decoder_parameter_count": sum(p.numel() for p in model.decoder.parameters()),
        "forward_resize_seconds_total": inference_seconds,
        "forward_resize_ms_per_query": 1000 * inference_seconds / len(rows),
        "wall_seconds_inference_metrics_saving": elapsed_all,
        "peak_allocated_gpu_mb": torch.cuda.max_memory_allocated(device) / 1024**2 if device.type == "cuda" else None,
        "timing_note": "Two warmup batches; forward+resize+sigmoid timed with CUDA sync; excludes preprocessing, transfer and metric/save. GPU peak includes resizing.",
        "save_probabilities": SAVE_PROBABILITIES,
    }
    write_json(OUT / "run.json", run)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    paper = summary["paper_protocol"]
    print(f"Theo protocol paper ({EVAL_SPLIT}): mIoU {100 * paper['miou_0.3']:.1f} | IoU_FG {100 * paper['fgiou_0.3']:.1f} | "
          f"AP {100 * paper['ap']:.1f}  (paper, CLIPSeg PC+ trên test: 43.4 | 54.7 | 76.7)")
    print(f"Forward+resize+sigmoid: {run['forward_resize_ms_per_query']:.2f} ms/query")


In [ ]:
if not STOPPED_EARLY and RUN_TEST:
    split_dir = OUT / "splits" / EVAL_SPLIT
    split_dir.mkdir(parents=True, exist_ok=True)
    for name in ["summary.json", "run.json", "sample_metrics.csv", "calibration_thresholds.csv",
                 "paper_threshold_stats.npz", "selected_tasks.json", "image_errors.json"]:
        if (OUT / name).is_file():
            shutil.copy2(OUT / name, split_dir / name)
    if MAX_QUERIES is None:
        assert summary["complete_official_split"], "Chưa chấm đủ split chính thức."
    paper = summary["paper_protocol"]
    print(f"Đã lưu {split_dir}: {paper['n_queries']} query, mIoU {100 * paper['miou_0.3']:.2f} | "
          f"IoU_FG {100 * paper['fgiou_0.3']:.2f} | AP {100 * paper['ap']:.2f}")


## 10. Xem hình và đóng gói

Trong Kaggle Output, tải `clipseg_e1.zip`. Giữ `run.json`, `selected_tasks.json`
và `sample_metrics.csv` cùng bảng số liệu để chạy lại đúng protocol.
Masks dùng ngưỡng cố định 0,3; ngưỡng calibration bổ sung chỉ được báo trong summary.


In [ ]:
for path in sorted((OUT / "visualizations").glob("*.png"))[:N_VISUALIZATIONS]:
    with Image.open(path) as image:
        display(image.copy())
archive = WORK_ROOT / f"clipseg_e4_{ARM.lower()}.zip"
training_artifacts = {"training_config.json", "training_history.csv", "train_tasks.json", "excluded_train_tasks.json", "checkpoints"}
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(OUT.rglob("*")):
        if not path.is_file():
            continue
        relative = path.relative_to(OUT)
        if not TRAIN_DECODER and relative.parts[0] in training_artifacts:
            continue
        package.write(path, arcname=relative.as_posix())
print("Result folder:", OUT)
print("Download:", archive)


## Nguồn và giới hạn

- [CLIPSeg / CVPR 2022](https://github.com/timojl/clipseg), [checkpoint HF](https://huggingface.co/CIDAS/clipseg-rd64-refined).
- [PhraseCutDataset](https://github.com/ChenyunWu/PhraseCutDataset): annotation, URL ảnh, split và quy ước polygon/mask.
- [CLIPSeg trong Transformers 4.55.4](https://huggingface.co/docs/transformers/v4.55.4/en/model_doc/clipseg).

Kích thước mask được đánh giá ở độ phân giải ảnh gốc, dùng rasterization PIL của PhraseCut API.
Repo CLIPSeg gốc dùng `skimage.draw.polygon2mask` và có preprocessing/evaluation khác;
số liệu notebook này cần ghi kèm checkpoint và protocol, không so trực tiếp như cùng một thiết lập.
Khi `TRAIN_DECODER=True`: decoder khởi tạo ngẫu nhiên, CLIP đóng băng, train trên PhraseCut+ (`TRAIN_MODE`).
Chọn `TRAIN_MODE="text_only"` cho ablation chỉ học text; chọn `pretrained_decoder` để fine-tune.
Google Drive/Visual Genome có thể gián đoạn; dataset input và checkpoint offline là đường dự phòng.
Khi Internet tắt, còn cần các dependency đã cài hoặc wheel đưa vào input.
